# 08 resultados R


In [ ]:
try:
    import google.colab
except ImportError as exc:
    raise RuntimeError('08 se ejecuta exclusivamente en Google Colab.') from exc
from pathlib import Path
import csv, hashlib, json, os, shutil, subprocess, sys, uuid
from datetime import datetime, timezone
from google.colab import drive
# Parámetros explícitos que proporciona el autor a partir de la salida de 06.
DRIVE_ROOT = Path('/content/drive/MyDrive/TESIS-EHBG-FACS')
REPORT_RELATIVE_PATH = ''  # comparison_runs/<bank_id>/comparison06_<...>/report
REPORT_CHECKSUMS_SHA256 = ''  # 64 caracteres, copiados de 06
SELECTION_REASON = ''  # ID de la corrida planificada/reanudada; no elegir por métricas

ANALYSIS_SOURCE = 'development06'  # 'final10' sólo para un release independiente sellado
FINAL_REPORT_RELATIVE_PATH = ''  # copiar de 10 la ruta relativa exacta que termina en /report
FINAL_REPORT_CHECKSUMS_SHA256 = ''


In [ ]:
ANALYSIS_CONFIG = {'schema_version': 'analysis-08-v1', 'role': 'development_only', 'analysis_mode': 'descriptive_only', 'source_export_schema': 'analysis-export-08-v1', 'unit_of_independence': 'geometry_group_sha256', 'run_selection': 'explicit_report_path_and_sha256_no_latest_or_best', 'solver_replica_aggregation': 'all_scheduled_seeds_mean_within_instance', 'geometry_aggregation': 'equal_instance_weight_within_geometry', 'missing_success': 'bounded_unknown_no_imputation', 'no_solution_success': 'zero_only_when_recorded_no_solution_and_no_routes', 'cost_analysis': 'conditional_conservative_feasible_with_explicit_denominators', 'cvar_aggregation': 'mean_of_per_plan_cvar_not_pooled_scenarios', 'reference_method': 'gurobi', 'confirmation': {'enabled_for_verified_final_export': True, 'accepted_export_schema': 'final-analysis-export-v1', 'accepted_role': 'confirmatory_holdout', 'requires_verified_population_protocol_freeze_test_chain': True, 'requires_frozen_analysis_r_sha256': True, 'primary_inference': 'stratified_geometry_weighted_hoeffding_mean_bound', 'multiplicity': 'holm_two_sided_familywise_0.05', 'development_results_are_not_confirmatory_evidence': True, 'enabled': False, 'enabled_explanation': 'Legacy development branch remains descriptive; final10 branch is selected explicitly and verifies an independent locked release.'}}
BANK_REFERENCE = {'bank_id': 'pilot00_280553a33c2ab594', 'protocol_version': 'pilot-00-v4', 'manifest_sha256': 'd8fe8d222f14acb9a32acfc62cdb105f3f364b85d47a4e8c96c7ef58c426fc02', 'acceptance_status': 'ready_for_method_development', 'role': 'development_only', 'validation_run_id': '20260917T021417Z_665a3a34', 'evidence': 'docs/NOTEBOOK_00_VALIDATION.md'}
ANALYSIS_CONFIG_SHA256 = 'fcb50845331cc84ab8ffb1e59efc0ffae7c72aa23b2d6c9326ac81e31a15b59b'


In [ ]:
REPORT_RELATIVE_PATH = (
    "comparison_runs/pilot00_280553a33c2ab594/"
    "comparison06_e52648d6c780761d5fae/report"
)

REPORT_CHECKSUMS_SHA256 = (
    "4163ae22430c8b0a45b0af2177195fc7fb0ffcaef719a2b572f9d56983e1a46f"
)

SELECTION_REASON = (
    "Consolidación de las ejecuciones del 21 de septiembre de 2026, "
    "seleccionadas por corresponder a la misma secuencia de verificación, "
    "sin selección por desempeño."
)

ANALYSIS_SOURCE = "development06"

## Verificación y staging


In [ ]:
if ANALYSIS_SOURCE == 'development06':
    if not REPORT_RELATIVE_PATH or len(REPORT_CHECKSUMS_SHA256) != 64 or not SELECTION_REASON.strip():
        raise ValueError('Fijar ruta, SHA-256 y motivo de selección del reporte 06 antes del análisis.')
    if ANALYSIS_CONFIG['analysis_mode'] != 'descriptive_only' or ANALYSIS_CONFIG['confirmation']['enabled']:
        raise ValueError('08 no implementa inferencia confirmatoria; requiere un protocolo y banco independientes.')
    drive.mount('/content/drive')
    def sha(path):
        digest = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        return digest.hexdigest()
    def read_json(path):
        return json.loads(Path(path).read_text(encoding='utf-8'))
    def safe(root, relative):
        relative = Path(relative)
        root = Path(root).resolve()
        if relative.is_absolute() or '..' in relative.parts or not relative.parts:
            raise ValueError('Ruta inválida en el reporte.')
        result = root / relative
        if not result.resolve().is_relative_to(root):
            raise ValueError('Ruta fuera del reporte.')
        return result
    REPORT = safe(DRIVE_ROOT, REPORT_RELATIVE_PATH)
    EXPECTED_PARENT = DRIVE_ROOT / 'comparison_runs' / BANK_REFERENCE['bank_id']
    if REPORT.name != 'report' or REPORT.parent.parent.resolve() != EXPECTED_PARENT.resolve():
        raise ValueError('Ruta ajena al banco fijado de desarrollo.')
    def verify_report(directory):
        if sha(directory / 'CHECKSUMS.json') != REPORT_CHECKSUMS_SHA256:
            raise ValueError('El SHA-256 del reporte no coincide con el seleccionado.')
        manifest = read_json(directory / 'CHECKSUMS.json')
        if not isinstance(manifest, dict) or not manifest:
            raise ValueError('Manifiesto vacío.')
        for name, digest in manifest.items():
            path = safe(directory, name)
            if not path.is_file() or sha(path) != digest:
                raise ValueError('Archivo del reporte alterado o ausente: ' + name)
        actual = {str(path.relative_to(directory)) for path in directory.rglob('*') if path.is_file()
                  and path.name != 'CHECKSUMS.json' and '__pycache__' not in path.relative_to(directory).parts}
        if actual != set(manifest):
            raise ValueError('Inventario del reporte incompleto o con archivos adicionales.')
        return manifest
    SOURCE_HASHES = verify_report(REPORT)
    contract = read_json(REPORT / 'analysis_export_contract.json')
    identity = read_json(REPORT / 'identity.json')
    validation = read_json(REPORT / 'validation.json')
    scope = read_json(REPORT / 'statistical_scope.json')
    protocol = read_json(REPORT / 'common_protocol.json')
    registry = read_json(REPORT / 'registry.json')
    for key, expected in {'schema_version': 'analysis-export-08-v1', 'role': 'development_only',
        'analysis_mode': 'descriptive_only', 'unit_of_independence': 'geometry_group_sha256',
        'bank_id': BANK_REFERENCE['bank_id'], 'bank_manifest_sha256': BANK_REFERENCE['manifest_sha256']}.items():
        if contract.get(key) != expected:
            raise ValueError('Contrato de exportación incompatible: ' + key)
    required = set(contract['required_files']) | {'analysis_export_contract.json', 'risk_diagnostics.csv',
        'budget_audit.csv', 'controlled_contrast_audit.csv', 'recourse_differences.csv', 'method_provenance.csv',
        'offline_costs.csv', 'teacher_costs.csv'}
    if not required <= set(SOURCE_HASHES):
        raise ValueError('Exportación 06 incompleta; regenerar con el contrato actual.')
    if (validation.get('status') != 'completed' or any(validation.get(k) is not True for k in
        ['bank_unchanged', 'source_results_unchanged', 'offline_manifests_unchanged', 'scenario_aggregates_reconciled'])
        or validation.get('ready_for_confirmatory_comparison') is not False
        or scope.get('ready_for_confirmatory') is not False or scope.get('no_pvalues') is not True):
        raise ValueError('06 carece de validación descriptiva completa.')
    if (contract['comparison_id'] != identity['comparison_id'] or
        contract['source_selection'] != identity['selected_runs'] or identity['selected_runs'] != validation['selected_runs'] or
        contract['expected_methods'] != identity['config']['expected_methods'] or
        contract['reference_method'] != ANALYSIS_CONFIG['reference_method'] or
        contract['policies'] != protocol['policies'] or contract['solver_seeds'] != protocol['solver_seeds']):
        raise ValueError('Identidad/protocolo/selección incompatibles.')
    # El hash del protocolo usa la serialización canónica publicada por el evaluador.
    canonical = json.dumps(protocol, sort_keys=True, separators=(',', ':'), ensure_ascii=False)
    if hashlib.sha256(canonical.encode()).hexdigest() != identity['comparison_id']:
        raise ValueError('Protocolo no corresponde al comparison_id.')
    if identity['config']['accepted_bank'] != BANK_REFERENCE:
        raise ValueError('El reporte usa otra aceptación de banco.')
    if len({r['instance_sha256'] for r in registry}) != len(registry) or not registry:
        raise ValueError('Registro vacío o duplicado.')
    WORK = Path('/content/analysis08_' + uuid.uuid4().hex[:12])
    INPUT = WORK / 'input'
    OUTPUT = WORK / 'output'
    INPUT.mkdir(parents=True)
    for name in SOURCE_HASHES:
        target = safe(INPUT, name)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(safe(REPORT, name), target)
    shutil.copy2(REPORT / 'CHECKSUMS.json', INPUT / 'CHECKSUMS.json')
    verify_report(INPUT)
    def write_csv(path, rows, fields):
        with Path(path).open('w', newline='', encoding='utf-8') as stream:
            writer = csv.DictWriter(stream, fieldnames=fields)
            writer.writeheader(); writer.writerows(rows)
    write_csv(INPUT / 'input_file_hashes.csv', [{'relative_path': k, 'sha256': v} for k, v in SOURCE_HASHES.items()], ['relative_path', 'sha256'])
    bridge = {key: contract[key] for key in ['schema_version', 'role', 'analysis_mode', 'comparison_id', 'reference_method']}
    bridge['report_id'] = REPORT.parent.name
    write_csv(INPUT / 'analysis_bridge.csv', [{'key': k, 'value': v} for k, v in bridge.items()], ['key', 'value'])
    for filename, field, values in [('expected_methods.csv', 'method', contract['expected_methods']),
        ('expected_policies.csv', 'policy', contract['policies']), ('expected_seeds.csv', 'solver_seed', contract['solver_seeds'])]:
        write_csv(INPUT / filename, [{field: str(value)} for value in values], [field])
    write_csv(INPUT / 'selected_runs.csv', [{'method': k, 'run_id': v} for k, v in identity['selected_runs'].items()], ['method', 'run_id'])
    Path('/content/.svrplab_analysis08_verified').write_text(REPORT_CHECKSUMS_SHA256)
    print('Reporte verificado:', REPORT.parent.name)
    print('N geometrías independientes:', len({r['geometry_group_sha256'] for r in registry}))
    print('Selección registrada:', SELECTION_REASON)


## R: agregación y publicación descriptiva


In [ ]:
if ANALYSIS_SOURCE == 'development06':
    R_SOURCE = '# 08: procesamiento descriptivo exclusivamente en Google Colab.\n# Entradas: copia íntegra de un reporte 06 verificado por el bootstrap Python.\n# R base: no instala paquetes ni transforma réplicas/escenarios en nuevas geometrías.\nargs <- commandArgs(trailingOnly = TRUE)\nif (Sys.getenv("SVRPLAB_COLAB_ANALYSIS") != "1" || length(args) != 2L ||\n    !dir.exists("/content") || !file.exists("/content/.svrplab_analysis08_verified")) {\n  stop("Ejecutar únicamente mediante el notebook 08 en Google Colab.")\n}\ninput <- normalizePath(args[[1]], mustWork = TRUE)\noutput <- args[[2]]\nif (dir.exists(output)) stop("La salida ya existe; conservar evidencia y usar un directorio nuevo.")\ndir.create(output, recursive = TRUE)\noptions(stringsAsFactors = FALSE, warn = 2, OutDec = ".")\nSys.setenv(TZ = "UTC")\nread <- function(name) {\n  path <- file.path(input, name)\n  if (!file.exists(path)) stop(paste("Falta entrada", name))\n  read.csv(path, colClasses = "character", check.names = FALSE, na.strings = "",\n           fileEncoding = "UTF-8", strip.white = FALSE)\n}\nwrite <- function(x, name) write.csv(x, file.path(output, name), row.names = FALSE, na = "", fileEncoding = "UTF-8")\nneed <- function(x, fields) if (!all(fields %in% names(x))) stop(paste("Columnas faltantes:", paste(setdiff(fields, names(x)), collapse = ", ")))\nnum <- function(x) {\n  y <- suppressWarnings(as.numeric(x))\n  if (any(!is.na(x) & (!is.finite(y) | is.na(y)))) stop("Métrica no finita/no numérica.")\n  y\n}\ntruth <- function(x) {\n  if (any(!is.na(x) & !x %in% c("True", "False", "true", "false"))) stop("Booleano inválido.")\n  !is.na(x) & x %in% c("True", "true")\n}\nkey <- function(x, fields) do.call(paste, c(x[fields], sep = "|"))\nmean_available <- function(x) if (all(is.na(x))) NA_real_ else mean(x[!is.na(x)])\nequal_num <- function(x, y) all((is.na(x) & is.na(y)) | (!is.na(x) & !is.na(y) & abs(x - y) <= 1e-8 * pmax(1, abs(y))))\nbridge <- read("analysis_bridge.csv")\nmeta <- setNames(bridge$value, bridge$key)\nif (meta[["schema_version"]] != "analysis-export-08-v1" || meta[["role"]] != "development_only" ||\n    meta[["analysis_mode"]] != "descriptive_only") stop("Alcance incompatible: no activar inferencia con desarrollo.")\nmethods <- read("expected_methods.csv")$method\npolicies <- read("expected_policies.csv")$policy\nseeds <- read("expected_seeds.csv")$solver_seed\nregistry <- read("registry.csv")\nneed(registry, c("instance_sha256", "instance_id", "geometry_group_sha256", "n_customers"))\nif (anyDuplicated(registry$instance_sha256) || anyNA(registry)) stop("Registro ambiguo.")\n# El hash de cada fuente ya fue validado; R vuelve a comprobarlo sin leer código fuente.\nhashes <- read("input_file_hashes.csv")\nobserved_hash <- vapply(hashes$relative_path, function(name) {\n  value <- system2("sha256sum", shQuote(file.path(input, name)), stdout = TRUE)\n  if (length(value) != 1L) stop("No se pudo verificar SHA-256.")\n  substr(value, 1, 64)\n}, character(1))\nif (!identical(unname(observed_hash), hashes$sha256)) stop("Una entrada cambió antes de R.")\nattempts <- read("attempts.csv")\nkeys <- c("instance_sha256", "policy", "method", "solver_seed")\nneed(attempts, c(keys, "geometry_group_sha256", "n_customers", "instance_id", "comparison_id",\n                "run_id", "evaluation_status", "solver_status", "structural_valid", "conservative_feasible",\n                "mean_travel_minutes", "cvar_travel_minutes", "scenario_strict_feasibility",\n                "mean_waiting_minutes", "mean_recourse_reorders", "search_total_seconds"))\nif (anyDuplicated(key(attempts, keys)) || anyNA(attempts[keys])) stop("Intentos duplicados/identidad ausente.")\nif (any(attempts$comparison_id != meta[["comparison_id"]])) stop("Identidades de comparación mezcladas.")\nselection <- read("selected_runs.csv")\nif (anyDuplicated(selection$method) || !all(selection$method %in% methods)) stop("Selección ambigua.")\nif (any(attempts$run_id != selection$run_id[match(attempts$method, selection$method)]) ||\n    anyNA(match(attempts$method, selection$method))) stop("Corrida no seleccionada.")\nreg <- registry[match(attempts$instance_sha256, registry$instance_sha256), ]\nfor (field in names(registry)) if (anyNA(reg[[field]]) || any(attempts[[field]] != reg[[field]])) stop("Identidad de instancia alterada.")\ngrid <- expand.grid(instance_sha256 = registry$instance_sha256, policy = policies, method = methods,\n                    solver_seed = seeds, stringsAsFactors = FALSE)\nif (!all(key(attempts, keys) %in% key(grid, keys))) stop("Intentos fuera de la cohorte.")\n# Todas las combinaciones previstas sobreviven, incluyendo métodos sin ninguna corrida.\nattempts$recorded <- TRUE\npanel <- merge(grid, attempts, by = keys, all.x = TRUE, sort = FALSE)\nreg <- registry[match(panel$instance_sha256, registry$instance_sha256), ]\nfor (field in setdiff(names(registry), "instance_sha256")) panel[[field]] <- reg[[field]]\npanel$recorded <- !is.na(panel$recorded)\npanel$evaluated <- !is.na(panel$evaluation_status) & panel$evaluation_status == "validated"\npanel$license_pending <- !is.na(panel$solver_status) & panel$solver_status == "license_pending"\npanel$failed <- panel$recorded & !panel$evaluated & !panel$license_pending\npanel$conservative <- panel$evaluated & truth(panel$structural_valid) & truth(panel$conservative_feasible)\nfor (field in c("mean_travel_minutes", "cvar_travel_minutes", "scenario_strict_feasibility",\n                "mean_waiting_minutes", "mean_recourse_reorders", "search_total_seconds")) panel[[field]] <- num(panel[[field]])\npanel$success <- ifelse(panel$evaluated, panel$scenario_strict_feasibility, NA_real_)\nno_solution <- panel$recorded & !panel$license_pending & !is.na(panel$evaluation_status) &\n  panel$evaluation_status == "no_solution" & !is.na(panel$structural_valid) & !truth(panel$structural_valid)\npanel$success[no_solution] <- 0\nif (any(panel$success < 0 | panel$success > 1, na.rm = TRUE)) stop("Éxito fuera de [0,1].")\npanel$duration_minutes <- panel$mean_travel_minutes + panel$mean_waiting_minutes\nwrite(panel, "attempt_panel_with_absences.csv")\n# Promediar semillas dentro de instancia; cada instancia pesa igual dentro de geometría.\ncase_keys <- c("instance_sha256", "policy", "method")\ngroups <- split(seq_len(nrow(panel)), key(panel, case_keys))\ncases <- do.call(rbind, lapply(groups, function(ii) {\n  x <- panel[ii, ]; expected <- nrow(x); known <- sum(!is.na(x$success))\n  feasible <- x$conservative\n  data.frame(x[1, c(case_keys, "geometry_group_sha256", "instance_id", "n_customers")],\n    expected = expected, recorded = sum(x$recorded), evaluated = sum(x$evaluated),\n    failed = sum(x$failed), missing = sum(!x$recorded), license_pending = sum(x$license_pending),\n    conservative_feasible = sum(feasible), success_known = known,\n    success = if (known == expected) mean(x$success) else NA_real_,\n    success_lower = sum(x$success, na.rm = TRUE) / expected,\n    success_upper = (sum(x$success, na.rm = TRUE) + expected - known) / expected,\n    travel = mean_available(x$mean_travel_minutes[feasible]),\n    cvar = mean_available(x$cvar_travel_minutes[feasible]),\n    duration = mean_available(x$duration_minutes[feasible]),\n    recourse_reorders = mean_available(x$mean_recourse_reorders[x$evaluated]),\n    search_seconds = mean_available(x$search_total_seconds[x$recorded]),\n    complete_feasible = all(feasible) && all(!is.na(x$mean_travel_minutes)) && all(!is.na(x$cvar_travel_minutes)),\n    complete_success = known == expected, independent_geometry_count = 1L)\n}))\nrownames(cases) <- NULL\nwrite(cases, "case_summary_R.csv")\n# Segunda implementación de agregados: detectar discrepancias respecto del reporte 06.\nexported <- read("joint_performance.csv")\nneed(exported, c(case_keys, "scenario_success_lower_bound", "scenario_success_upper_bound",\n                "mean_travel_conditional_conservative_feasible", "expected_attempts", "failed_attempts", "missing_attempts"))\nif (anyDuplicated(key(exported, case_keys)) || !setequal(key(cases, case_keys), key(exported, case_keys))) stop("Cohorte de joint_performance incompatible.")\nexported <- exported[match(key(cases, case_keys), key(exported, case_keys)), ]\npairs <- c(success_lower = "scenario_success_lower_bound", success_upper = "scenario_success_upper_bound",\n           travel = "mean_travel_conditional_conservative_feasible", expected = "expected_attempts",\n           failed = "failed_attempts", missing = "missing_attempts")\nfor (field in names(pairs)) if (!equal_num(cases[[field]], num(exported[[pairs[[field]]]]))) stop(paste("Discrepancia R/06:", field))\ngeom_keys <- c("geometry_group_sha256", "policy", "method")\ngeometry <- do.call(rbind, lapply(split(seq_len(nrow(cases)), key(cases, geom_keys)), function(ii) {\n  x <- cases[ii, ]\n  data.frame(x[1, geom_keys], n_customers = paste(sort(unique(x$n_customers)), collapse = ";"),\n    instances = nrow(x), expected = sum(x$expected), recorded = sum(x$recorded), evaluated = sum(x$evaluated),\n    failed = sum(x$failed), missing = sum(x$missing), license_pending = sum(x$license_pending),\n    conservative_feasible = sum(x$conservative_feasible),\n    success_lower = mean(x$success_lower), success_upper = mean(x$success_upper),\n    success = if (all(x$complete_success)) mean(x$success) else NA_real_,\n    travel = mean_available(x$travel), cvar = mean_available(x$cvar), duration = mean_available(x$duration),\n    recourse_reorders = mean_available(x$recourse_reorders), search_seconds = mean_available(x$search_seconds),\n    complete_feasible = all(x$complete_feasible), complete_success = all(x$complete_success),\n    independent_geometry_count = 1L)\n}))\nrownames(geometry) <- NULL\nwrite(geometry, "geometry_summary_R.csv")\n# Los contrastes causales necesitan la auditoría de 06. Conservar todas sus exclusiones;\n# no reconstruir un efecto neuronal/feromona a partir de nombres de métodos.\nfor (name in c("paired_geometry_costs.csv", "paired_geometry_success.csv", "gaps.csv",\n               "solution_diagnostics.csv", "risk_diagnostics.csv", "budget_audit.csv",\n               "controlled_contrast_audit.csv", "recourse_differences.csv", "method_provenance.csv",\n               "offline_costs.csv", "teacher_costs.csv")) write(read(name), name)\n# Salida larga accesible y tabla Markdown con denominadores al lado de cada costo.\nformat_num <- function(x) ifelse(is.na(x), "—", formatC(x, digits = 2, format = "f"))\nlines <- c("# Resultados descriptivos de desarrollo · procesamiento R", "",\n  paste("Reporte 06:", meta[["report_id"]], "· comparison_id:", meta[["comparison_id"]]), "",\n  "N es el número de geometrías. Semillas, escenarios, políticas y folds no aumentan N.",\n  "Los intervalos [inferior, superior] son cotas por información ausente, NO intervalos de confianza.",\n  "Viaje/CVaR/duración se condicionan a factibilidad conservadora; no constituyen un ranking con cobertura desigual.",\n  "Duración = viaje + espera; el costo c+Q de dos etapas sigue sin evaluarse.", "")\nfor (policy in policies) {\n  lines <- c(lines, paste("## Política", policy), "",\n    "| Geometría · n | Método | Evaluados/previstos | Factibles/previstos | Viaje | CVaR por plan | Duración | Éxito [cotas] | Fallos | Ausentes/licencia |",\n    "|---|---|---:|---:|---:|---:|---:|---:|---:|---:|")\n  for (geom in registry$geometry_group_sha256[!duplicated(registry$geometry_group_sha256)]) for (method in methods) {\n    x <- geometry[geometry$geometry_group_sha256 == geom & geometry$policy == policy & geometry$method == method, ]\n    if (nrow(x) != 1L) stop("Geometría/método ambiguo.")\n    lines <- c(lines, sprintf("| %s · %s | %s | %d/%d | %d/%d | %s | %s | %s | [%.3f, %.3f] | %d | %d/%d |",\n      substr(geom, 1, 12), x$n_customers, method, x$evaluated, x$expected, x$conservative_feasible, x$expected,\n      format_num(x$travel), format_num(x$cvar), format_num(x$duration), x$success_lower, x$success_upper,\n      x$failed, x$missing, x$license_pending))\n  }\n  lines <- c(lines, "")\n}\nlines <- c(lines, "No se calcularon p-valores, bootstrap poblacional, intervalos poblacionales ni selección de la mejor corrida.",\n  "Las cotas Gurobi son del sustituto conservador: gaps.csv nunca las resta del viaje medio estocástico.",\n  "Las figuras usan un orden fijo de métodos; los fallos conservan su fila y denominador.")\nwriteLines(lines, file.path(output, "results_for_thesis.md"), useBytes = TRUE)\n# Figuras estáticas reproducibles, sin barras de error que aparenten réplica poblacional.\nfor (policy in policies) for (geom in unique(geometry$geometry_group_sha256)) {\n  x <- geometry[geometry$geometry_group_sha256 == geom & geometry$policy == policy, ]\n  x <- x[match(methods, x$method), ]; y <- seq_along(methods)\n  draw <- function() {\n    par(mfrow = c(1, 2), mar = c(5, 11, 4, 1), las = 1, cex = .75)\n    labels <- paste0(x$method, "  [", x$conservative_feasible, "/", x$expected, "]")\n    maximum <- max(c(1, x$travel, x$cvar), na.rm = TRUE)\n    plot(x$travel, y, xlim = c(0, maximum * 1.08), ylim = c(.5, length(y) + .5), yaxt = "n", pch = 19,\n         xlab = "Minutos: media y CVaR por plan", ylab = "", main = "Costo condicional [factibles/previstos]")\n    axis(2, at = y, labels = labels); points(x$cvar, y, pch = 1, col = "#0072B2")\n    text(rep(0, sum(is.na(x$travel))), y[is.na(x$travel)], "sin medición", pos = 4, col = "#666666")\n    legend("bottomright", c("Viaje medio", "CVaR medio"), pch = c(19, 1), col = c("black", "#0072B2"), bty = "n")\n    plot(x$success_lower, y, xlim = c(0, 1), ylim = c(.5, length(y) + .5), yaxt = "n", pch = 19,\n         xlab = "Éxito por intento previsto; línea = desconocimiento", ylab = "", main = "Éxito y cobertura de todas las semillas")\n    segments(x$success_lower, y, x$success_upper, y, col = "#D55E00", lwd = 2)\n    mtext(paste("n =", x$n_customers[[1]], "· N = 1 geometría ·", substr(geom, 1, 12), "·", policy), side = 3, line = 2.5, cex = .8)\n  }\n  stem <- paste0("geometry_", substr(geom, 1, 12), "_", policy)\n  pdf(file.path(output, paste0(stem, ".pdf")), width = 14, height = 8, family = "sans"); draw(); dev.off()\n  png(file.path(output, paste0(stem, ".png")), width = 4200, height = 2400, res = 300); draw(); dev.off()\n}\n# Revalidar fuentes tras el procesamiento: ninguna lectura se convierte en edición.\nfor (i in seq_len(nrow(hashes))) {\n  value <- system2("sha256sum", shQuote(file.path(input, hashes$relative_path[[i]])), stdout = TRUE)\n  if (substr(value[[1]], 1, 64) != hashes$sha256[[i]]) stop("Fuente cambió durante R.")\n}\nwriteLines(capture.output(sessionInfo()), file.path(output, "sessionInfo.txt"))\nwrite(data.frame(check = c("expected_cohort_preserved", "source_hashes_verified_before_after", "R_06_aggregates_reconciled",\n                          "no_population_inference", "geometry_is_independence_unit", "no_best_run_selection"),\n                 passed = TRUE), "processing_checks.csv")\ncat("R completó tablas/figuras descriptivas. Falta el sellado final del notebook 08.\\n")\n'
    R_SCRIPT = WORK / 'process_results_08.R'
    R_SCRIPT.write_text(R_SOURCE, encoding='utf-8')
    if shutil.which('Rscript') is None:
        subprocess.run(['apt-get', '-qq', 'update'], check=True)
        subprocess.run(['apt-get', '-qq', 'install', '-y', 'r-base-core'], check=True)
    analysis_env = dict(os.environ, SVRPLAB_COLAB_ANALYSIS='1')
    result = subprocess.run(['Rscript', '--vanilla', str(R_SCRIPT), str(INPUT), str(OUTPUT)],
                            env=analysis_env, text=True, capture_output=True)
    (WORK / 'R_stdout.log').write_text(result.stdout)
    (WORK / 'R_stderr.log').write_text(result.stderr)
    print(result.stdout)
    if result.returncode:
        print(result.stderr)
        raise RuntimeError('R se detuvo; no se publica un reporte validado. Revisar logs en ' + str(WORK))


In [ ]:
if ANALYSIS_SOURCE == 'development06':
    # Sellar sólo después de que R concilie agregados e integridad.
    verify_report(REPORT)
    processing_id = 'analysis08_' + hashlib.sha256((REPORT_CHECKSUMS_SHA256 + sha(R_SCRIPT) +
        json.dumps(ANALYSIS_CONFIG, sort_keys=True)).encode()).hexdigest()[:20]
    processing_identity = {'schema_version': 'analysis-08-v1', 'processing_id': processing_id,
        'input_report_relative_path': REPORT_RELATIVE_PATH, 'input_report_checksums_sha256': REPORT_CHECKSUMS_SHA256,
        'comparison_id': identity['comparison_id'], 'source_selection': identity['selected_runs'],
        'selection_reason': SELECTION_REASON, 'config': ANALYSIS_CONFIG, 'r_script_sha256': sha(R_SCRIPT),
        'created_utc': datetime.now(timezone.utc).isoformat(), 'ready_for_confirmatory_comparison': False,
        'runtime_validation': 'R_06_aggregates_reconciled_and_source_hashes_verified'}
    (OUTPUT / 'identity.json').write_text(json.dumps(processing_identity, indent=2, ensure_ascii=False) + '\n')
    shutil.copy2(R_SCRIPT, OUTPUT / R_SCRIPT.name)
    for filename in ['R_stdout.log', 'R_stderr.log']:
        shutil.copy2(WORK / filename, OUTPUT / filename)
    for filename in ['analysis_export_contract.json', 'common_protocol.json', 'registry.json', 'input_environments.json']:
        shutil.copy2(INPUT / filename, OUTPUT / filename)
    (OUTPUT / 'input_manifest.json').write_text(json.dumps(SOURCE_HASHES, indent=2) + '\n')
    manifest = {str(path.relative_to(OUTPUT)): sha(path) for path in sorted(OUTPUT.rglob('*')) if path.is_file()}
    (OUTPUT / 'CHECKSUMS.json').write_text(json.dumps(manifest, indent=2, sort_keys=True) + '\n')
    RUN_SUFFIX = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:6]
    DESTINATION = DRIVE_ROOT / 'analysis_runs' / BANK_REFERENCE['bank_id'] / processing_id / RUN_SUFFIX
    if DESTINATION.exists():
        raise FileExistsError('No se reemplaza un análisis existente.')
    shutil.copytree(OUTPUT, DESTINATION)
    for name, digest in manifest.items():
        if sha(DESTINATION / name) != digest:
            raise ValueError('Copia Drive incompleta: ' + name)
    if sha(DESTINATION / 'CHECKSUMS.json') != sha(OUTPUT / 'CHECKSUMS.json'):
        raise ValueError('Manifiesto de salida inconsistente.')
    print('Procesamiento descriptivo completo y sellado:', DESTINATION)
    print('SHA-256 de CHECKSUMS:', sha(DESTINATION / 'CHECKSUMS.json'))
    from IPython.display import display, Markdown, Image
    display(Markdown((DESTINATION / 'results_for_thesis.md').read_text()))
    for figure in sorted(DESTINATION.glob('*.png')):
        display(Image(filename=str(figure), width=1100))


## Separación de desarrollo y confirmación


## Rama final10: confirmación sólo sobre un release independiente


In [ ]:
FINAL_MODULES = {'__init__.py': '"""Infraestructura experimental de la tesis HBG-FACS."""\n', 'pilot.py': '"""Notebook 00: datos, auditoría, CRN y evaluación independiente del solver.\n\nAdaptación del generador de tráfico de SVRPBench (MIT, Ahmed Heakl, 2024).\nVer docs/SVRPBench-LICENSE.txt y docs/NOTEBOOK_00.md para las diferencias.\nNo contiene entrenamiento HBG/AM/POMO ni demuestra superioridad estadística.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport importlib\nimport importlib.metadata\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport platform\nimport random\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport urllib.request\nimport warnings\nfrom contextlib import contextmanager\nfrom dataclasses import asdict, dataclass\n\nimport numpy as np\n\nUPSTREAM_SHA = \'9fd5737ae598e1667ae94907f9c45224c501c84b\'\nHF_REVISION = \'4a79ade38f663edaefa62e34e7c2ea330a2e15da\'\nROWS_SHA = \'9c055992902ddf9c6133c438bc35e09623d59b0a376b623cbadd3375bbd36147\'\nSOURCE_FILES = (\'city.py\', \'constants.py\', \'common.py\', \'time_windows_generator.py\',\n                \'travel_time_generator.py\', \'real_twcvrp.py\')\nSOURCE_HASHES = {\'city.py\': \'c55ef26cfbd24fa705afd6223aa45cf53094168ba02d95552562c9a66dc09e9e\', \'constants.py\': \'28cca2aecf61bad8e4fb75dc6807f18f200998eb8a767a62e425c50641f25758\', \'common.py\': \'8400c0e767c189987ccf26dee9688117ee2deb4e14f3c8ce516dd30e70ea5f5b\', \'time_windows_generator.py\': \'90b19f40fdef99604345bf5935ceb40b54f10e5d869c77974bcef736a26d2afb\', \'travel_time_generator.py\': \'a0f6251321bcf053f251c58dd6d7ce207b7c2f66595444fdface1d54127d6e5e\', \'real_twcvrp.py\': \'3530823c42ef6480b6d0aa63d3535b8e8313e20d82b904647653cf0c5da9de07\'}\nPROTOCOL_VERSION = \'pilot-00-v4\'\nWINDOW_POLICY = \'official_type_width_conditioned_reachability_v1\'\nJOINT_WINDOW_POLICY = \'joint_official_draws_proven_rejection_v1\'\nLEGACY_WINDOW_POLICY = \'balanced_insertion_mean_time_projection_v1\'\n# Candidato rechazado en Colab, comunicado por el autor; NO banco aceptado.\nCOLAB_REJECTED_V3_SHA = {50: \'e7390617fa2471dc120f03182eeeb9d6ea081a2ce44b8e1b390fd095e980531d\'}\n# Instancias validadas (master_seed, n) -> sha256. Coinciden con Python 3.12 + versiones\n# fijadas y con el entorno Colab (Python 3.13, numpy 2.1.3, scikit-learn 1.6.1).\n# Garantizan reproducibilidad por contenido, no por versión de biblioteca.\nREFERENCE_INSTANCE_SHA = {\n    (20260916, 50): \'17dc741073c2b524d3bfbd97912fd4abd9ab033267faf2bc629265de9fb1f0f9\',\n    (20260916, 100): \'540bb7187a90ef06f56f90c4a5e0cb0438bfc902d2077975fb3faa7fe914319f\',\n    (20260916, 200): \'fb3e12f54bd1003e00ed1910601935fe714aa561ef2feadeae90221592644445\',\n    (20260916, 300): \'3e7cf85cf9c76b4fb10b73d013ffbc4b6ee7017d777ad1c093928be45877eec8\',\n}\nLEGACY_V2_REFERENCE_SHA = {\n    50: \'cb57f1068a3c7b173dbd087154f8276a8ba98df7cb45cc542fc8a3f5ef12b94f\',\n    100: \'3e70717e9302373d12ddf25c613ef3c84522ba6c4f8a735c3368684db60eccff\',\n    200: \'387948482f58d7f1e8d6c109f5edcbcc59ea318660b166119e76dc5b3f6c422c\',\n    300: \'d2c4b2457c08cf9fa7981fa13a85bd0637ea914ef9eb281f62acdc96f3f3b231\',\n}\n\n\ndef canonical_bytes(value):\n    return json.dumps(value, sort_keys=True, separators=(\',\', \':\'),\n                      ensure_ascii=False, allow_nan=False).encode(\'utf-8\')\n\n\ndef digest(value):\n    return hashlib.sha256(canonical_bytes(value)).hexdigest()\n\n\ndef file_hash(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for chunk in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + \'.tmp\')\n    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + \'\\n\')\n    temporary.replace(path)\n\n\ndef seed_for(*parts):\n    return int.from_bytes(hashlib.sha256(canonical_bytes(parts)).digest()[:8], \'little\')\n\n\n@contextmanager\ndef legacy_seed(seed):\n    """Aísla las RNG globales que usa el generador oficial."""\n    py, np_state = random.getstate(), np.random.get_state()\n    random.seed(seed)\n    np.random.seed(seed % 2**32)\n    try:\n        yield\n    finally:\n        random.setstate(py)\n        np.random.set_state(np_state)\n\n\ndef download(url, destination):\n    """Descarga atómica, tiempo limitado; no ejecuta contenido descargado."""\n    destination = Path(destination)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    request = urllib.request.Request(url, headers={\'User-Agent\': \'SVRPLab-thesis/1.0\'})\n    with urllib.request.urlopen(request, timeout=90) as response:\n        payload = response.read()\n    temporary = destination.with_suffix(destination.suffix + \'.download\')\n    temporary.write_bytes(payload)\n    temporary.replace(destination)\n    return destination\n\n\ndef fetch_upstream(root):\n    root = Path(root)\n    for name in SOURCE_FILES:\n        path = root / \'vrp_bench\' / name\n        if not path.exists():\n            download(f\'https://raw.githubusercontent.com/yehias21/svrpbench/{UPSTREAM_SHA}/vrp_bench/{name}\', path)\n    if not (root / \'LICENSE\').exists():\n        download(f\'https://raw.githubusercontent.com/yehias21/svrpbench/{UPSTREAM_SHA}/LICENSE\', root / \'LICENSE\')\n    for name, expected in SOURCE_HASHES.items():\n        if file_hash(root / \'vrp_bench\' / name) != expected:\n            raise ValueError(f\'Código upstream modificado: {name}\')\n    # Hashes verificados; el commit completo fija las URLs.\n    return {str(p.relative_to(root)): file_hash(p)\n            for p in sorted(root.rglob(\'*\')) if p.is_file() and p.suffix == \'.py\'}\n\n\ndef load_catalog(cache):\n    """Parquet fijado; fallback oficial Viewer sólo si coincide el hash de 560 filas.\n\n    La revisión del Viewer no se puede seleccionar: se identifica como snapshot\n    verificado por contenido, nunca como descarga de la revisión fijada.\n    """\n    cache = Path(cache)\n    cache.mkdir(parents=True, exist_ok=True)\n    snapshot = cache / \'catalog_rows.json\'\n    meta_file = cache / \'catalog_provenance.json\'\n    if snapshot.exists():\n        rows = json.loads(snapshot.read_text())\n        meta = json.loads(meta_file.read_text()) if meta_file.exists() else {\'transport\': \'local_verified_snapshot\'}\n    else:\n        rows = None\n        try:\n            from huggingface_hub import hf_hub_download\n            import pyarrow.parquet as pq\n            path = hf_hub_download(repo_id=\'MBZUAI/svrp-bench\',\n                filename=\'data/test-00000-of-00001.parquet\', repo_type=\'dataset\',\n                revision=HF_REVISION, local_dir=str(cache / \'hf\'))\n            rows = pq.read_table(path).to_pylist()\n            meta = {\'transport\': \'pinned_parquet\', \'revision\': HF_REVISION,\n                    \'file_sha256\': file_hash(path)}\n            # El hash fijado se obtuvo del Viewer; si la representación Parquet difiere\n            # (tipos numéricos), se intenta el Viewer antes de declarar el catálogo distinto.\n            if digest(sorted(rows, key=lambda r: (r[\'subset_name\'], r[\'instance_id\']))) != ROWS_SHA:\n                warnings.warn(\'Parquet no coincide con el hash fijado; comprobando snapshot oficial Viewer.\')\n                rows = None\n        except Exception as exc:\n            warnings.warn(f\'Parquet no disponible ({type(exc).__name__}); comprobando snapshot oficial Viewer.\')\n        if rows is None:\n            rows = []\n            for offset in range(0, 560, 100):\n                url = (\'https://datasets-server.huggingface.co/rows?dataset=MBZUAI%2Fsvrp-bench\'\n                       f\'&config=default&split=test&offset={offset}&length=100\')\n                path = download(url, cache / f\'viewer_{offset}.json\')\n                page = json.loads(path.read_text())\n                if page.get(\'partial\') or page[\'num_rows_total\'] != 560:\n                    raise ValueError(\'El catálogo cambió o está incompleto. Auditar antes de continuar.\')\n                for item in page[\'rows\']:\n                    if item.get(\'truncated_cells\'):\n                        raise ValueError(\'Viewer truncó celdas. Se necesita el Parquet completo.\')\n                    rows.append(item[\'row\'])\n            meta = {\'transport\': \'viewer_content_verified\', \'revision\': None,\n                    \'snapshot_observed\': \'2026-09-16\', \'requested_revision\': HF_REVISION}\n    rows = sorted(rows, key=lambda r: (r[\'subset_name\'], r[\'instance_id\']))\n    if len(rows) != 560 or digest(rows) != ROWS_SHA:\n        raise ValueError(\'Hash del catálogo distinto. No seleccionar datos nuevos silenciosamente.\')\n    meta.update(dataset=\'MBZUAI/svrp-bench\', split=\'test\', rows_sha256=ROWS_SHA,\n                rows=len(rows), role=\'development_only_never_final_test\')\n    write_json(snapshot, rows)\n    write_json(meta_file, meta)\n    return rows, meta\n\n\ndef upstream_modules(root):\n    directory = str((Path(root) / \'vrp_bench\').resolve())\n    if directory not in sys.path:\n        sys.path.insert(0, directory)\n    for name in (\'constants\', \'city\', \'common\', \'time_windows_generator\'):\n        mod = importlib.import_module(name)\n        if Path(mod.__file__).resolve().parent != Path(directory):\n            raise RuntimeError(f\'Módulo {name} procede de otra ruta; reinicie el entorno.\')\n    return sys.modules[\'common\'], sys.modules[\'time_windows_generator\']\n\n\n@dataclass(frozen=True)\nclass Instance:\n    instance_id: str\n    locations: np.ndarray\n    demands: np.ndarray\n    capacities: np.ndarray\n    windows: np.ndarray\n    appear_times: np.ndarray\n    provenance: dict\n\n    @property\n    def n(self):\n        return len(self.demands) - 1\n\n    def payload(self):\n        return {\'instance_id\': self.instance_id, \'depot_indices\': [0],\n                **{key: getattr(self, key).tolist() for key in\n                   (\'locations\', \'demands\', \'capacities\', \'windows\', \'appear_times\')},\n                \'provenance\': self.provenance}\n\n    @property\n    def sha(self):\n        return digest(self.payload())\n\n\ndef freeze_instance(payload):\n    if payload.get(\'depot_indices\') != [0]:\n        raise ValueError(\'Este piloto admite exactamente el depósito 0.\')\n    arrays = {}\n    for field in (\'locations\', \'demands\', \'capacities\', \'windows\', \'appear_times\'):\n        arrays[field] = np.array(payload[field], dtype=np.float64, copy=True)\n        arrays[field].setflags(write=False)\n    x = Instance(payload[\'instance_id\'], **arrays, provenance=payload[\'provenance\'])\n    n = x.n + 1\n    if n < 2 or x.locations.shape != (n, 2) or x.windows.shape != (n, 2):\n        raise ValueError(\'Dimensiones inválidas.\')\n    if x.demands.shape != (n,) or x.appear_times.shape != (n,) or x.capacities.ndim != 1:\n        raise ValueError(\'Dimensiones inválidas en demanda/flota/arribos.\')\n    if not all(np.isfinite(v).all() for v in arrays.values()):\n        raise ValueError(\'Datos no finitos.\')\n    if len(x.capacities) == 0 or np.any(x.capacities <= 0) or np.any(x.demands < 0):\n        raise ValueError(\'Capacidades/demandas inválidas.\')\n    if x.demands[0] != 0 or np.any(x.appear_times < 0) or np.any(x.windows[:, 0] > x.windows[:, 1]):\n        raise ValueError(\'Depósito, aparición o ventanas inválidos.\')\n    return x\n\n\ndef make_instance(n, rows, root, master_seed=20260916):\n    common, tw = upstream_modules(root)\n    subset = f\'twvrp_{n}_single_depot\'\n    candidates = sorted([r for r in rows if r[\'subset_name\'] == subset], key=lambda r: r[\'instance_id\'])\n    seed = seed_for(master_seed, \'geometry\', n)\n    if candidates:\n        row = candidates[0]  # elección anterior a ejecutar cualquier solver\n        raw = row\n        xy, demand, appears = row[\'locations\'], row[\'demands\'], row[\'appear_times\']\n        caps = row[\'vehicle_capacities\']\n        if len(caps) == 1:\n            caps = caps * int(row[\'num_vehicles\'])\n        if len(caps) != int(row[\'num_vehicles\']):\n            raise ValueError(\'Flota inconsistente; no reparar de manera implícita.\')\n        origin = {\'kind\': \'hf_geometry_demands_fleet_plus_derived_windows\',\n                  \'subset_name\': subset, \'source_instance_id\': int(row[\'instance_id\']),\n                  \'source_row_sha256\': digest(row)}\n    else:\n        if n != 300:\n            raise ValueError(f\'Tamaño {n} sin datos publicados y sin derivación aprobada.\')\n        from threadpoolctl import threadpool_limits\n        with legacy_seed(seed), threadpool_limits(limits=1):\n            base = common.generate_base_instance(n, (1000, 1000), max(1, n // 50), 1, (0, 100), False)\n        xy, demand, appears = base[\'locations\'].tolist(), base[\'demands\'].tolist(), base[\'appear_time\'].tolist()\n        caps = None\n        raw = {\'locations\': xy, \'demands\': demand, \'appear_times\': appears, \'seed\': seed}\n        origin = {\'kind\': \'derived_official_base_generator\', \'geometry_seed\': seed,\n                  \'generator\': \'common.generate_base_instance\', \'num_cities\': max(1, n // 50)}\n    if len(xy) != n + 1:\n        raise ValueError(\'Número de nodos inconsistente; no inferir depósitos por demanda cero.\')\n    windows_seed = seed_for(master_seed, \'time_windows\', n)\n    # Replica la regla de real_twcvrp: tipo elegido con random.randint(0,1),\n    # no la proporción 60/40 descrita en el artículo. Se preserva esa diferencia.\n    with legacy_seed(windows_seed):\n        windows = [tw.sample_time_window(random.randint(0, 1), a) for a in appears]\n    windows[0] = (0, 1440)\n    # El dataset original cuantiza ventanas a uint16; se mantiene en derivados.\n    windows = np.array(windows, dtype=np.uint16).tolist()\n    if caps is None:\n        # Equivalente a plt.hist(..., bins=24, range=(0,1440)) de get_num_vehicles.\n        count = int(np.histogram(np.asarray(windows).ravel(), bins=24, range=(0, 1440))[0].max())\n        caps = [float(sum(demand))] * max(1, count)\n    origin.update(upstream_commit=UPSTREAM_SHA, windows_seed=windows_seed,\n                  windows=\'derived_NOT_original_HF_windows\', time_unit=\'minute\',\n                  coordinate_unit=\'source_map_unit\', service_minutes=0,\n                  customer_ids=\'1..n_including_zero_demands\', depot_ids=[0],\n                  capacity_modified=False)\n    payload = dict(instance_id=f\'dev_tw_n{n}_i0\', depot_indices=[0], locations=xy,\n                   demands=demand, capacities=caps, windows=windows, appear_times=appears,\n                   provenance=origin)\n    instance = freeze_instance(payload)\n    expected = REFERENCE_INSTANCE_SHA.get((master_seed, n))\n    if expected and instance.sha != expected:\n        raise ValueError(f\'Instancia n={n} distinta de la validada (probable cambio de numpy/scikit-learn). \'\n                         \'No se sustituye en silencio: active STRICT_PINS = True en la celda 1 y ejecute de nuevo.\')\n    return instance, raw\n\n\ndef distances(x):\n    # Euclídea float64: coherente con VRPSolverBase; Location.distance del generador\n    # trunca a entero. Se declara la diferencia, no se mezcla entre métodos.\n    return np.linalg.norm(x.locations[:, None, :] - x.locations[None, :, :], axis=-1)\n\n\ndef audit_instance(x, spec=None, certificate=None):\n    """Cotas necesarias con viaje mínimo d/v; pasan a ser prueba sólo si fallan.\n\n    Incluye apertura y regreso: un cliente puede ser alcanzable antes del cierre\n    y aun así hacer imposible el retorno, incluso visitándolo solo.\n    """\n    spec = spec or Traffic()\n    d = distances(x)\n    earliest = np.maximum.reduce([d[0] / spec.velocity, x.windows[:, 0], x.appear_times])\n    impossible = np.flatnonzero(earliest > x.windows[:, 1] + 1e-9)\n    impossible = [int(j) for j in impossible if j != 0]\n    return_impossible = [int(j) for j in range(1, x.n + 1)\n                         if earliest[j] + d[j, 0] / spec.velocity > x.windows[0, 1] + 1e-9]\n    oversized = np.flatnonzero(x.demands > x.capacities.max()).tolist()\n    capacity_ok = bool(x.capacities.sum() >= x.demands.sum())\n    certified = certificate is not None and verify_certificate(x, certificate, spec)[\'strict_feasible\']\n    return {\'instance_id\': x.instance_id, \'n_customers\': x.n, \'n_depots\': 1,\n            \'available_vehicles\': len(x.capacities), \'total_demand\': float(x.demands.sum()),\n            \'zero_demand_customers\': int(np.sum(x.demands[1:] == 0)),\n            \'capacity_min\': float(x.capacities.min()),\n            \'capacity_nonbinding_per_vehicle\': bool(np.all(x.capacities >= x.demands.sum())),\n            \'unreachable_by_window_lower_bound\': impossible,\n            \'impossible_depot_return_lower_bound\': return_impossible,\n            \'oversized_demands\': oversized,\n            \'aggregate_capacity_sufficient\': capacity_ok,\n            \'proven_infeasible_by_lower_bound\': bool(impossible or return_impossible or oversized or not capacity_ok),\n            \'source_kind\': x.provenance[\'kind\'], \'sha256\': x.sha,\n            \'feasibility_certified\': bool(certified),\n            \'certificate_scope\': \'nominal_mean_only\' if certified else \'none\'}\n\n\n@dataclass(frozen=True)\nclass Traffic:\n    velocity: float = 1.0\n    noise_scale: float = 1.0\n    accident_scale: float = 1.0\n    bucket_minutes: float = 1.0\n    mode: str = \'stochastic\'  # stochastic / no_noise / nominal_mean (no escenarios)\n\n    def __post_init__(self):\n        if not all(math.isfinite(v) for v in (self.velocity, self.bucket_minutes, self.noise_scale, self.accident_scale)) or self.velocity <= 0 or self.bucket_minutes <= 0 or self.noise_scale < 0 or self.accident_scale < 0:\n            raise ValueError(\'Parámetros de tráfico fuera del dominio.\')\n        if self.mode not in (\'stochastic\', \'no_noise\', \'nominal_mean\'):\n            raise ValueError(\'Modo de tráfico desconocido.\')\n\n\ndef normal_pdf(t, mu, sigma):\n    return math.exp(-0.5 * ((t - mu) / sigma)**2) / (sigma * math.sqrt(2 * math.pi))\n\n\ndef traffic_parameters(t, dist, spec):\n    peak = normal_pdf(t, 480, 90) + normal_pdf(t, 1020, 90)\n    base = 0.25 * (0.5 + 2 * peak) * (1 - math.exp(-dist / 50))\n    mu, sigma = 0.1 * peak, (0.3 + 0.2 * peak) * spec.noise_scale\n    rate = 0.05 * normal_pdf(t, 1260, 120) * spec.accident_scale\n    return base, mu, sigma, rate\n\n\ndef expected_time(dist, t, spec):\n    if dist == 0 or spec.mode == \'no_noise\':\n        return dist / spec.velocity\n    base, mu, sigma, rate = traffic_parameters(t, dist, spec)\n    return dist / spec.velocity + base * math.exp(mu + sigma * sigma / 2) + rate * 75\n\n\ndef travel_time(dist, t, i, j, instance_id, scenario_seed, spec):\n    if dist == 0 or i == j or spec.mode == \'no_noise\':\n        return dist / spec.velocity\n    if spec.mode == \'nominal_mean\':\n        return expected_time(dist, t, spec)\n    base, mu, sigma, rate = traffic_parameters(t, dist, spec)\n    # Flujo RNG local: método, orden de evaluación y ruta no figuran en la clave.\n    # La misma arista/minuto/escenario recibe los mismos números primitivos.\n    bucket = math.floor(t / spec.bucket_minutes)\n    rng = np.random.default_rng(seed_for(\'traffic-v1\', instance_id, int(scenario_seed), i, j, bucket))\n    factor = math.exp(mu + sigma * rng.standard_normal())\n    accidents = rng.poisson(rate)\n    accident_delay = float(rng.uniform(30, 120, size=accidents).sum())\n    return dist / spec.velocity + base * factor + accident_delay\n\n\ndef scenario_seeds(master_seed=20260916, instance_id=\'\', split=\'evaluation\', count=256):\n    if split not in (\'train\', \'selection\', \'evaluation\') or count < 1:\n        raise ValueError(\'Split/count inválidos.\')\n    return [seed_for(master_seed, \'scenario\', instance_id, split, k) for k in range(count)]\n\n\ndef validate_routes(x, routes):\n    if not isinstance(routes, (list, tuple)) or len(routes) != len(x.capacities):\n        raise ValueError(\'Una lista por vehículo, incluidas listas vacías; nodos sin depósito.\')\n    flat = []\n    for v, route in enumerate(routes):\n        if not isinstance(route, (list, tuple)):\n            raise ValueError(\'Cada ruta debe ser una lista de índices enteros.\')\n        if any(isinstance(j, bool) or not isinstance(j, (int, np.integer)) or j < 1 or j > x.n for j in route):\n            raise ValueError(\'Cliente fuera de rango o depósito dentro de la ruta.\')\n        if sum(x.demands[j] for j in route) > x.capacities[v] + 1e-9:\n            raise ValueError(\'Capacidad excedida en una ruta.\')\n        flat.extend(route)\n    if sorted(flat) != list(range(1, x.n + 1)):\n        raise ValueError(\'Todos los clientes, incluso demanda cero, deben aparecer exactamente una vez.\')\n    return True\n\n\ndef _reference_schedule(x, route, d, spec):\n    """Horario nominal sin las ventanas antiguas, con apariciones y regreso."""\n    t, i, starts = 0., 0, {}\n    for j in route:\n        t = max(t + expected_time(d[i, j], t, spec), float(x.appear_times[j]))\n        starts[j] = t\n        i = j\n    return starts, t + expected_time(d[i, 0], t, spec) if route else 0.\n\n\ndef derive_feasible_windows(parent, spec=Traffic(), margin_fraction=.25):\n    """Archivo v2, sólo regresión/diagnóstico; no usar para el banco de la tesis.\n\n    Inserción balanceada: clientes de mayor a menor distancia al depósito;\n    entre inserciones que respetan capacidad y horizonte, elige menor tiempo\n    de retorno nominal, luego menor incremento, vehículo y posición. No usa\n    ventanas de clientes ni escenarios ni resultados de los métodos.\n\n    Proyecta cada apertura al intervalo más cercano que sitúa el horario\n    testigo en el interior de la ventana, conservando su duración entera.\n    El margen predefinido es 1/4 de la duración, limitado por [0, horizonte].\n    Si no logra construir/certificar, falla: no prueba infactibilidad global.\n    """\n    if not 0 <= margin_fraction < .5:\n        raise ValueError(\'Margen fuera de [0, .5).\')\n    if parent.windows[0, 0] != 0:\n        raise ValueError(\'Este protocolo exige apertura del depósito en 0.\')\n    nominal = Traffic(**(asdict(spec) | {\'mode\': \'nominal_mean\'}))\n    d, horizon = distances(parent), float(parent.windows[0, 1])\n    routes = [[] for _ in parent.capacities]\n    loads, returns = np.zeros(len(routes)), np.zeros(len(routes))\n    for j in sorted(range(1, parent.n + 1), key=lambda k: (-d[0, k], k)):\n        choices = []\n        for v, route in enumerate(routes):\n            if loads[v] + parent.demands[j] > parent.capacities[v] + 1e-9:\n                continue\n            for pos in range(len(route) + 1):\n                candidate = route[:pos] + [j] + route[pos:]\n                _, end = _reference_schedule(parent, candidate, d, nominal)\n                if end <= horizon + 1e-9:\n                    choices.append(((end, end - returns[v], v, pos), v, candidate, end))\n        if not choices:\n            raise ValueError(f\'No se pudo construir testigo para cliente {j}; no cambiar flota/horizonte. \'\n                             \'No constituye prueba de infactibilidad global.\')\n        _, v, routes[v], returns[v] = min(choices)\n        loads[v] += parent.demands[j]\n    validate_routes(parent, routes)\n    windows = parent.windows.copy()\n    for route in routes:\n        starts, _ = _reference_schedule(parent, route, d, nominal)\n        for j, t in starts.items():\n            width = float(parent.windows[j, 1] - parent.windows[j, 0])\n            if width < 2 or width != int(width) or width > horizon or horizon != int(horizon):\n                raise ValueError(\'Proyección requiere duración entera >=2 y <= horizonte entero.\')\n            margin = min(margin_fraction * width, t, horizon - t)\n            lower = max(0, math.ceil(t - width + margin - 1e-9))\n            upper = min(int(horizon - width), math.floor(t - margin + 1e-9))\n            if lower > upper:\n                raise ValueError(\'No hay proyección entera admisible; no ensanchar ventanas.\')\n            opening = min(max(int(parent.windows[j, 0]), lower), upper)\n            windows[j] = [opening, opening + width]\n    shifts = windows[1:, 0] - parent.windows[1:, 0]\n    transformation = {\'policy\': LEGACY_WINDOW_POLICY, \'margin_fraction\': margin_fraction,\n                      \'parent_instance_sha256\': parent.sha, \'nominal_traffic\': asdict(nominal),\n                      \'changed_windows\': int(np.count_nonzero(shifts)),\n                      \'mean_absolute_shift_minutes\': float(np.abs(shifts).mean()),\n                      \'max_absolute_shift_minutes\': float(np.abs(shifts).max()),\n                      \'window_widths_preserved\': True, \'geometry_demands_fleet_preserved\': True,\n                      \'uses_evaluation_scenarios\': False}\n    payload = parent.payload()\n    payload.update(instance_id=parent.instance_id + \'_fw_v2\', windows=windows.tolist(),\n                   provenance=dict(parent.provenance, kind=\'nominal_feasible_derived_windows\',\n                                   parent_kind=parent.provenance[\'kind\'],\n                                   windows=\'derived_projected_NOT_original_HF_windows\',\n                                   transformation=transformation))\n    instance = freeze_instance(payload)\n    if (parent.sha == REFERENCE_INSTANCE_SHA.get((20260916, parent.n))\n            and spec == Traffic() and margin_fraction == .25\n            and instance.sha != LEGACY_V2_REFERENCE_SHA[parent.n]):\n        raise ValueError(\'Derivado v2 distinto de la referencia; banco rechazado. \'\n                         \'Compruebe código/entorno; STRICT_PINS permite reproducir el entorno validado.\')\n    certificate = {\'scope\': \'nominal_mean_only\', \'not_a_competitive_baseline\': True,\n                   \'not_solver_input\': True, \'policy\': LEGACY_WINDOW_POLICY,\n                   \'parent_instance_sha256\': parent.sha, \'instance_sha256\': instance.sha,\n                   \'traffic\': asdict(nominal), \'routes\': routes}\n    certificate[\'verification\'] = verify_certificate(instance, certificate, spec)\n    return instance, certificate\n\n\ndef verify_certificate(x, certificate, spec=Traffic()):\n    """Recalcula el testigo con el evaluador común; no confía en un booleano guardado."""\n    nominal = Traffic(**(asdict(spec) | {\'mode\': \'nominal_mean\'}))\n    if certificate.get(\'instance_sha256\') != x.sha or certificate.get(\'traffic\') != asdict(nominal):\n        raise ValueError(\'Certificado corresponde a otra instancia o parametrización de tráfico.\')\n    if certificate.get(\'scope\') != \'nominal_mean_only\':\n        raise ValueError(\'Alcance de certificado no reconocido.\')\n    if certificate.get(\'policy\') == WINDOW_POLICY:\n        verify_conservative_routes(x, certificate[\'routes\'], nominal,\n                                   certificate[\'time_scale\'])\n    result = simulate(x, certificate[\'routes\'], 0, nominal, \'fixed_order\', trace=True)\n    if not result[\'strict_feasible\']:\n        raise ValueError(\'El testigo no satisface ventanas/regreso nominales; banco rechazado.\')\n    slack = [x.windows[e[\'to\'], 1] - e[\'service_start\'] for e in result[\'events\'] if e[\'to\'] != 0]\n    return {\'strict_feasible\': True, \'used_vehicles\': result[\'used_vehicles\'],\n            \'minimum_customer_slack_minutes\': float(min(slack)),\n            \'maximum_return_minutes\': float(max(e[\'arrival\'] for e in result[\'events\'] if e[\'to\'] == 0)),\n            \'travel_minutes\': result[\'travel_minutes\'], \'waiting_minutes\': result[\'waiting_minutes\']}\n\n\ndef nominal_upper_matrix(x, spec=Traffic(), time_scale=1000):\n    """Cota superior de la MEDIA por arco para toda hora, no del ruido aleatorio.\n\n    Cada densidad normal <=1/(sigma*sqrt(2*pi)); sustituir ambos picos\n    por sus máximos acota base, mu, sigma y tasa de accidentes. Se redondea\n    hacia arriba a milésimas de minuto para el certificador entero.\n    """\n    if not isinstance(time_scale, int) or isinstance(time_scale, bool) or time_scale < 1:\n        raise ValueError(\'time_scale debe ser entero positivo.\')\n    d = distances(x)\n    peak_max = 2 / (90 * math.sqrt(2 * math.pi))\n    sigma_max = (.3 + .2 * peak_max) * spec.noise_scale\n    delay = .25 * (.5 + 2 * peak_max) * (1 - np.exp(-d / 50))\n    delay *= math.exp(.1 * peak_max + sigma_max**2 / 2)\n    delay += 75 * .05 * spec.accident_scale / (120 * math.sqrt(2 * math.pi))\n    upper = d / spec.velocity + (delay if spec.mode != \'no_noise\' else 0)\n    upper[d == 0] = 0\n    return np.ceil(upper * time_scale).astype(np.int64)\n\n\ndef official_customer_types(parent, root):\n    """Reproduce etiquetas latentes sin cambiar el generador ni los hashes v1."""\n    _, tw = upstream_modules(root)\n    types, windows = [], []\n    with legacy_seed(parent.provenance[\'windows_seed\']):\n        for appearance in parent.appear_times:\n            kind = random.randint(0, 1)\n            types.append(kind)\n            windows.append(tw.sample_time_window(kind, appearance))\n    windows[0] = (0, 1440)\n    if not np.array_equal(np.asarray(windows, dtype=np.uint16), parent.windows):\n        raise ValueError(\'La reconstrucción de tipos no coincide con las ventanas v1.\')\n    types[0] = -1  # depósito, no cliente residencial/comercial\n    return types\n\n\ndef make_conditioned_candidate(parent, root, spec=Traffic(), max_draws_per_customer=10000,\n                               time_scale=1000, joint_attempt=0):\n    """Intento 0 reproduce v3; los siguientes remuestrean TODAS las ventanas.\n\n    Todo horario nuevo es una salida íntegra del generador oficial (uint16),\n    del mismo tipo y duración que v1. Nunca se desplaza, recorta ni ensancha.\n    Tipo, duración, geometría y flota son fijos para todos los intentos.\n    La distribución resultante está condicionada por alcanzabilidad individual;\n    esto NO prueba factibilidad global, que requiere un certificado separado.\n    """\n    if max_draws_per_customer < 1:\n        raise ValueError(\'Presupuesto de muestras inválido.\')\n    if not isinstance(joint_attempt, int) or isinstance(joint_attempt, bool) or joint_attempt < 0:\n        raise ValueError(\'Índice de remuestreo conjunto inválido.\')\n    _, tw = upstream_modules(root)\n    types = official_customer_types(parent, root)\n    upper = nominal_upper_matrix(parent, spec, time_scale)\n    windows, ledger = parent.windows.copy(), []\n    horizon = math.floor(parent.windows[0, 1] * time_scale)\n    def acceptable(j, pair):\n        start = max(int(upper[0, j]), math.ceil(max(pair[0], parent.appear_times[j]) * time_scale))\n        return start <= math.floor(pair[1] * time_scale) and start + int(upper[j, 0]) <= horizon\n    for j in range(1, parent.n + 1):\n        if int(upper[0, j]) + int(upper[j, 0]) > horizon:\n            raise ValueError(f\'Geometría n={parent.n}, cliente {j}: no pasa el filtro individual. \'\n                             \'No cambiar velocidad, flota ni horizonte automáticamente.\')\n        old = parent.windows[j].tolist()\n        width = old[1] - old[0]\n        if joint_attempt == 0 and acceptable(j, old):\n            ledger.append({\'customer\': j, \'type\': types[j], \'draw\': 0, \'seed\': None,\n                           \'window\': old, \'accepted\': True, \'origin\': \'preserved_v1\'})\n            continue\n        for draw in range(1, max_draws_per_customer + 1):\n            seed = (seed_for(parent.provenance[\'windows_seed\'], WINDOW_POLICY, j, draw) if joint_attempt == 0\n                    else seed_for(parent.provenance[\'windows_seed\'], JOINT_WINDOW_POLICY, joint_attempt, j, draw))\n            with legacy_seed(seed):\n                proposed = np.asarray(tw.sample_time_window(types[j], parent.appear_times[j]),\n                                      dtype=np.uint16).tolist()\n            width_ok = proposed[1] - proposed[0] == width\n            reachable = acceptable(j, proposed)\n            accepted = width_ok and reachable\n            ledger.append({\'customer\': j, \'type\': types[j], \'draw\': draw, \'seed\': seed,\n                           \'window\': proposed, \'accepted\': accepted, \'width_matches\': width_ok,\n                           \'individual_screen_passed\': reachable, \'origin\': \'official_generator\'})\n            if accepted:\n                windows[j] = proposed\n                break\n        else:\n            raise ValueError(f\'Presupuesto de muestreo agotado en n={parent.n}, cliente {j}; \'\n                             \'candidato no certificado. No desplazar ventanas como fallback.\')\n    shifts = windows[1:, 0] - parent.windows[1:, 0]\n    change = {\'policy\': WINDOW_POLICY, \'parent_instance_sha256\': parent.sha,\n              \'changed_windows\': int(np.count_nonzero(shifts)),\n              \'mean_absolute_shift_minutes\': float(np.abs(shifts).mean()),\n              \'max_absolute_shift_minutes\': float(np.abs(shifts).max()),\n              \'window_widths_preserved\': True, \'customer_types_preserved\': True,\n              \'geometry_demands_fleet_preserved\': True,\n              \'conditioned_distribution_not_original_benchmark\': True,\n              \'uses_evaluation_scenarios\': False, \'time_scale\': time_scale,\n              \'nominal_traffic\': asdict(Traffic(**(asdict(spec) | {\'mode\': \'nominal_mean\'}))),\n              \'sampling_ledger_sha256\': digest(ledger)}\n    if joint_attempt:\n        change.update(joint_policy=JOINT_WINDOW_POLICY, joint_attempt=joint_attempt)\n    payload = parent.payload()\n    suffix = f\'_joint{joint_attempt:03d}\' if joint_attempt else \'\'\n    payload.update(instance_id=parent.instance_id + \'_cw_v3\' + suffix, windows=windows.tolist(),\n        provenance=dict(parent.provenance, kind=\'official_conditioned_windows\',\n                        windows=\'official_draws_conditioned_NOT_original_HF_windows\',\n                        customer_types=types,\n                        geometry_group_sha256=digest({\'locations\': parent.locations.tolist(), \'depot_indices\': [0]}),\n                        transformation=change))\n    candidate = freeze_instance(payload)\n    if (joint_attempt == 0 and parent.sha == REFERENCE_INSTANCE_SHA.get((20260916, parent.n))\n            and spec == Traffic() and time_scale == 1000 and parent.n in COLAB_REJECTED_V3_SHA\n            and candidate.sha != COLAB_REJECTED_V3_SHA[parent.n]):\n        raise ValueError(\'El intento 0 difiere del candidato v3 observado en Colab. Revisar código/entorno; \'\n                         \'no sustituir silenciosamente el caso rechazado.\')\n    return candidate, ledger\n\n\ndef verify_conservative_routes(x, routes, spec=Traffic(), time_scale=1000):\n    """Chequeo determinista independiente de los horarios que devuelve OR-Tools."""\n    validate_routes(x, routes)\n    upper = nominal_upper_matrix(x, spec, time_scale)\n    horizon = math.floor(x.windows[0, 1] * time_scale)\n    for route in routes:\n        t, i = 0, 0\n        for j in route:\n            t = max(t + int(upper[i, j]), math.ceil(max(x.windows[j, 0], x.appear_times[j]) * time_scale))\n            if t > math.floor(x.windows[j, 1] * time_scale):\n                raise ValueError(\'Testigo viola una ventana en el verificador conservador.\')\n            i = j\n        if t + int(upper[i, 0]) > horizon:\n            raise ValueError(\'Testigo viola el retorno en el verificador conservador.\')\n    return True\n\n\ndef fleet_compatibility_diagnostic(x, spec=Traffic(), tolerance=1e-8):\n    """Cota de vehículos por clientes mutuamente incompatibles, incluso sin ruido.\n\n    E_i=max(apertura_i,aparición_i,d_0i/v), L_i=min(cierre_i,H-d_i0/v).\n    Si E_i+d_ij/v>L_j, i no puede preceder a j, tampoco con visitas intermedias\n    (desigualdad triangular). Si ambas direcciones fallan, no comparten vehículo.\n    Una clique de k clientes exige al menos k vehículos. La clique hallada es\n    una cota, no necesariamente máxima; pasar este chequeo no prueba factibilidad.\n    """\n    if not math.isfinite(tolerance) or tolerance < 0:\n        raise ValueError(\'Tolerancia inválida.\')\n    d = distances(x) / spec.velocity\n    earliest = np.maximum.reduce([d[0], x.windows[:, 0], x.appear_times])\n    latest = np.minimum(x.windows[:, 1], x.windows[0, 1] - d[:, 0])\n    impossible_single = [j for j in range(1, x.n + 1) if earliest[j] > latest[j] + tolerance]\n    bad_order = earliest[:, None] + d > latest[None, :] + tolerance\n    incompatible = bad_order & bad_order.T\n    np.fill_diagonal(incompatible, False)\n    incompatible[0, :] = incompatible[:, 0] = False\n    neighbors = {i: set(np.flatnonzero(incompatible[i]).tolist()) for i in range(1, x.n + 1)}\n    best = []\n    for first in sorted(neighbors, key=lambda i: (-len(neighbors[i]), i)):\n        clique, pending = [first], set(neighbors[first])\n        while pending:\n            j = min(pending, key=lambda i: (-len(neighbors[i] & pending), i))\n            clique.append(j)\n            pending &= neighbors[j]\n        if len(clique) > len(best):\n            best = clique\n    proof_pairs = []\n    for a, i in enumerate(best):\n        for j in best[a + 1:]:\n            ij = float(earliest[i] + d[i, j] - latest[j])\n            ji = float(earliest[j] + d[j, i] - latest[i])\n            if ij <= tolerance or ji <= tolerance:\n                raise AssertionError(\'Cota de clique no verificada.\')\n            proof_pairs.append({\'i\': i, \'j\': j, \'i_before_j_impossible_by_minutes\': ij,\n                                \'j_before_i_impossible_by_minutes\': ji})\n    fleet_insufficient = len(best) > len(x.capacities)\n    return {\'instance_sha256\': x.sha, \'scope\': \'optimistic_free_flow_necessary_conditions\',\n            \'velocity\': spec.velocity, \'tolerance_minutes\': tolerance,\n            \'available_vehicles\': len(x.capacities),\n            \'individually_impossible_clients\': impossible_single,\n            \'vehicles_lower_bound\': len(best), \'mutually_incompatible_clients\': best,\n            \'clique_is_proven_maximum\': False, \'pair_witnesses\': proof_pairs,\n            \'proves_insufficient_fleet\': fleet_insufficient,\n            \'proves_infeasibility\': bool(impossible_single or fleet_insufficient),\n            \'passing_is_not_feasibility_certificate\': True}\n\n\ndef certification_model_data(x, spec=Traffic(), time_scale=1000, optimistic=False):\n    """Datos enteros reproducibles; el modelo optimista es una RELAJACIÓN.\n\n    Bajar cada tránsito y abrir hacia afuera los dominios preserva toda solución\n    continua al tomar floor(t*S). INFEASIBLE en esta relajación prueba imposibilidad\n    real. La cota superior nominal hace lo contrario: sólo certifica testigos.\n    """\n    if not isinstance(time_scale, int) or isinstance(time_scale, bool) or time_scale < 1:\n        raise ValueError(\'time_scale debe ser entero positivo.\')\n    opening = np.maximum(x.windows[:, 0], x.appear_times)\n    if optimistic:\n        # Margen hacia afuera para que error flotante no endurezca la relajación.\n        eps = 1e-8\n        matrix = np.floor(np.maximum(0, distances(x) / spec.velocity - eps) * time_scale).astype(np.int64)\n        lower = np.floor(np.maximum(0, opening - eps) * time_scale).astype(np.int64)\n        upper = np.ceil((x.windows[:, 1] + eps) * time_scale).astype(np.int64)\n        horizon = math.ceil((x.windows[0, 1] + eps) * time_scale)\n    else:\n        matrix = nominal_upper_matrix(x, spec, time_scale)\n        lower = np.ceil(opening * time_scale).astype(np.int64)\n        upper = np.floor(x.windows[:, 1] * time_scale).astype(np.int64)\n        horizon = math.floor(x.windows[0, 1] * time_scale)\n    return {\'formulation\': \'mandatory_multiple_circuit_time_windows_v1\',\n            \'scope\': \'optimistic_free_flow_relaxation\' if optimistic else \'conservative_nominal_mean\',\n            \'instance_sha256\': x.sha, \'time_scale\': time_scale, \'fleet\': len(x.capacities),\n            \'matrix\': matrix.tolist(), \'lower\': lower.tolist(), \'upper\': upper.tolist(),\n            \'horizon\': horizon, \'capacity\': \'nonbinding_checked_before_solve\'}\n\n\ndef solve_certification_model(x, data, seconds, workers):\n    """CP-SAT de factibilidad, sin objetivo ni eliminación de clientes.\n\n    No se distinguen vehículos: esto elimina simetrías y requiere capacidad no\n    vinculante. Cada circuito pasa por el depósito; el número de salidas <= K.\n    """\n    from ortools.sat.python import cp_model\n    model = cp_model.CpModel()\n    lower, upper, horizon = data[\'lower\'], data[\'upper\'], data[\'horizon\']\n    matrix = data[\'matrix\']\n    times = {}\n    for j in range(1, x.n + 1):\n        if lower[j] > upper[j]:\n            model.add(False)\n        times[j] = model.new_int_var(min(lower[j], upper[j]), upper[j], f\'t{j}\')\n    arcs, variables = [], {}\n    # Incluso un nodo aislado queda explícitamente en el circuito; su self-loop\n    # es falso, por lo que no se permite omitirlo de la solución.\n    for j in range(1, x.n + 1):\n        arcs.append((j, j, model.new_constant(0)))\n    for i in range(x.n + 1):\n        for j in range(x.n + 1):\n            if i == j:\n                continue\n            start = 0 if i == 0 else lower[i]\n            deadline = horizon if j == 0 else upper[j]\n            if start + matrix[i][j] > deadline:\n                continue\n            arc = model.new_bool_var(f\'a{i}_{j}\')\n            variables[i, j] = arc\n            arcs.append((i, j, arc))\n            if i == 0:\n                model.add(times[j] >= matrix[i][j]).only_enforce_if(arc)\n            elif j == 0:\n                model.add(times[i] + matrix[i][j] <= horizon).only_enforce_if(arc)\n            else:\n                model.add(times[j] >= times[i] + matrix[i][j]).only_enforce_if(arc)\n    model.add_multiple_circuit(arcs)\n    model.add(sum(arc for (i, j), arc in variables.items() if i == 0) <= len(x.capacities))\n    validation = model.validate()\n    if validation:\n        return None, {\'status_name\': \'MODEL_INVALID\', \'validation_error\': validation,\n                      \'seconds\': 0.0, \'solution_returned\': False, \'model_sha256\': digest(data)}\n    solver = cp_model.CpSolver()\n    solver.parameters.max_time_in_seconds = float(seconds)\n    solver.parameters.num_search_workers = workers\n    solver.parameters.random_seed = 0\n    started = time.perf_counter()\n    status = solver.solve(model)\n    attempt = {\'status_name\': solver.status_name(status), \'cp_sat_status\': int(status),\n               \'seconds\': time.perf_counter() - started, \'model_sha256\': digest(data),\n               \'solution_returned\': status in (cp_model.FEASIBLE, cp_model.OPTIMAL),\n               \'scope\': data[\'scope\'], \'time_scale\': data[\'time_scale\'],\n               \'workers\': workers, \'random_seed\': 0, \'limit_seconds\': seconds,\n               \'ortools_version\': importlib.metadata.version(\'ortools\'),\n               \'response_stats\': solver.response_stats()}\n    if not attempt[\'solution_returned\']:\n        return None, attempt\n    selected = [(i, j) for (i, j), arc in variables.items() if solver.boolean_value(arc)]\n    successor = {i: j for i, j in selected if i != 0}\n    routes = []\n    seen = set()\n    for first in sorted(j for i, j in selected if i == 0):\n        route, j = [], first\n        while j:\n            if j in seen:\n                raise ValueError(\'Circuito devuelto no es una partición de los clientes.\')\n            seen.add(j); route.append(j); j = successor[j]\n        routes.append(route)\n    routes += [[] for _ in range(len(x.capacities) - len(routes))]\n    validate_routes(x, routes)\n    return routes, attempt\n\n\ndef certify_candidate(x, spec=Traffic(), seconds_per_strategy=120, time_scale=1000, progress=False):\n    """Prueba negativa optimista o testigo positivo conservador, nunca reparar datos.\n\n    INFEASIBLE conservador y UNKNOWN no permiten rechazar/remuestrear. El modelo\n    sólo cubre flotas cuyas capacidades individuales son >= demanda total.\n    """\n    if not math.isfinite(seconds_per_strategy) or seconds_per_strategy < 1:\n        raise ValueError(\'Presupuesto de certificación inválido.\')\n    diagnostic = fleet_compatibility_diagnostic(x, spec)\n    base = {\'instance_sha256\': x.sha, \'diagnostic\': diagnostic,\n            \'is_proof_of_infeasibility\': False, \'attempts\': []}\n    if progress:\n        print(f"Cota necesaria de vehículos: >= {diagnostic[\'vehicles_lower_bound\']}; "\n              f"disponibles: {len(x.capacities)}.", flush=True)\n    if diagnostic[\'proves_infeasibility\']:\n        return None, dict(base, status=\'proven_infeasible_free_flow_bound\', is_proof_of_infeasibility=True)\n    if np.any(x.capacities < x.demands.sum()):\n        raise ValueError(\'CP-SAT 00 requiere capacidad no vinculante en cada vehículo; no ignorar esta restricción.\')\n    workers = min(8, os.cpu_count() or 1)\n    attempts = base[\'attempts\']\n    for stage, optimistic in enumerate((True, False), 1):\n        data = certification_model_data(x, spec, time_scale, optimistic)\n        if progress:\n            print(f"[{stage}/2] CP-SAT {data[\'scope\']}; límite {seconds_per_strategy} s.", flush=True)\n        routes, attempt = solve_certification_model(x, data, seconds_per_strategy, workers)\n        attempts.append(attempt)\n        if progress:\n            print(f"[{stage}/2] {attempt[\'status_name\']} en {attempt[\'seconds\']:.2f} s.", flush=True)\n        if attempt[\'status_name\'] == \'MODEL_INVALID\':\n            return None, dict(base, status=\'certifier_model_invalid\')\n        if optimistic and attempt[\'status_name\'] == \'INFEASIBLE\':\n            return None, dict(base, status=\'proven_infeasible_optimistic_model\', is_proof_of_infeasibility=True)\n        if routes is None:\n            continue\n        try:\n            verify_conservative_routes(x, routes, spec, time_scale)\n        except ValueError:\n            attempt[\'conservative_witness_verified\'] = False\n            if not optimistic:\n                raise  # un testigo del modelo superior debe pasar el verificador\n            continue\n        attempt[\'conservative_witness_verified\'] = True\n        certificate = {\'scope\': \'nominal_mean_only\', \'policy\': WINDOW_POLICY,\n                       \'instance_sha256\': x.sha,\n                       \'parent_instance_sha256\': x.provenance[\'transformation\'][\'parent_instance_sha256\'],\n                       \'traffic\': asdict(Traffic(**(asdict(spec) | {\'mode\': \'nominal_mean\'}))),\n                       \'routes\': routes, \'time_scale\': time_scale,\n                       \'not_solver_input\': True, \'not_a_competitive_baseline\': True,\n                       \'certifier\': \'OR-Tools CP-SAT feasibility; independently verified upper-mean witness\',\n                       \'ortools_version\': importlib.metadata.version(\'ortools\'), \'attempts\': attempts}\n        certificate[\'verification\'] = verify_certificate(x, certificate, spec)\n        return certificate, dict(base, status=\'certified_nominal\')\n    return None, dict(base, status=\'unknown_no_certificate\')\n\n\ndef validate_sampling_ledger(x, ledger):\n    """Integridad y cobertura de aceptación; reproducción oficial en celda Colab."""\n    change = x.provenance[\'transformation\']\n    if change.get(\'policy\') != WINDOW_POLICY or digest(ledger) != change[\'sampling_ledger_sha256\']:\n        raise ValueError(\'Política o registro de muestreo no corresponde al candidato.\')\n    accepted = [entry for entry in ledger if entry[\'accepted\']]\n    if sorted(entry[\'customer\'] for entry in accepted) != list(range(1, x.n + 1)):\n        raise ValueError(\'Debe haber exactamente una ventana aceptada por cliente.\')\n    for entry in accepted:\n        j = entry[\'customer\']\n        if entry[\'type\'] != x.provenance[\'customer_types\'][j] or not np.array_equal(entry[\'window\'], x.windows[j]):\n            raise ValueError(\'Ventana/tipo aceptado no coincide con la instancia.\')\n    return True\n\n\ndef proven_rejection(x, report, spec=Traffic(), time_scale=1000):\n    """Sólo dos fuentes admitidas de prueba negativa; timeout nunca es rechazo.\n\n    La clique se recalcula aquí. Para CP-SAT se verifica que INFEASIBLE corresponde\n    a la relajación reconstruida; su prueba depende del solver, no de reejecutarlo.\n    """\n    if report.get(\'instance_sha256\') != x.sha or report.get(\'is_proof_of_infeasibility\') is not True:\n        return False\n    if report.get(\'status\') == \'proven_infeasible_free_flow_bound\':\n        diagnostic = fleet_compatibility_diagnostic(x, spec)\n        return diagnostic[\'proves_infeasibility\'] and report.get(\'diagnostic\') == diagnostic\n    if report.get(\'status\') == \'proven_infeasible_optimistic_model\':\n        fingerprint = digest(certification_model_data(x, spec, time_scale, optimistic=True))\n        return any(a.get(\'scope\') == \'optimistic_free_flow_relaxation\'\n                   and a.get(\'status_name\') == \'INFEASIBLE\' and a.get(\'cp_sat_status\') == 3\n                   and a.get(\'model_sha256\') == fingerprint and not a.get(\'solution_returned\')\n                   for a in report.get(\'attempts\', []))\n    return False\n\n\ndef validate_admission_history(x, history, spec=Traffic(), time_scale=1000):\n    """No puede faltar un intento ni descartarse un resultado desconocido."""\n    if history.get(\'policy\') != JOINT_WINDOW_POLICY or not history.get(\'attempts\'):\n        raise ValueError(\'Falta historial de admisión conjunta.\')\n    records = history[\'attempts\']\n    if len(records) > history[\'max_joint_candidates\']:\n        raise ValueError(\'Se excedió el presupuesto preregistrado de candidatos.\')\n    for index, record in enumerate(records):\n        candidate = freeze_instance(record[\'candidate\'])\n        transformation = candidate.provenance[\'transformation\']\n        if (record[\'joint_attempt\'] != index or transformation.get(\'joint_attempt\', 0) != index\n                or (index > 0 and transformation.get(\'joint_policy\') != JOINT_WINDOW_POLICY)\n                or transformation[\'parent_instance_sha256\'] != x.provenance[\'transformation\'][\'parent_instance_sha256\']):\n            raise ValueError(\'Intentos omitidos, desordenados o de otro origen.\')\n        for field in (\'locations\', \'demands\', \'capacities\', \'appear_times\'):\n            if not np.array_equal(getattr(candidate, field), getattr(x, field)):\n                raise ValueError(\'Geometría, demandas, flota o apariciones cambiaron entre intentos.\')\n        if (candidate.provenance[\'customer_types\'] != x.provenance[\'customer_types\']\n                or not np.array_equal(candidate.windows[:, 1] - candidate.windows[:, 0],\n                                      x.windows[:, 1] - x.windows[:, 0])\n                or not np.array_equal(candidate.windows[0], x.windows[0])):\n            raise ValueError(\'Tipos, duraciones o depósito cambiaron entre intentos.\')\n        validate_sampling_ledger(candidate, record[\'ledger\'])\n        if index < len(records) - 1:\n            if record[\'certificate\'] is not None or not proven_rejection(candidate, record[\'certification\'], spec, time_scale):\n                raise ValueError(\'Se saltó un candidato sin prueba de imposibilidad.\')\n        else:\n            if (candidate.sha != x.sha or record[\'certification\'].get(\'status\') != \'certified_nominal\'\n                    or record[\'certification\'].get(\'instance_sha256\') != x.sha):\n                raise ValueError(\'La instancia final no coincide con el primer certificado.\')\n            verify_certificate(candidate, record[\'certificate\'], spec)\n    return True\n\n\ndef admit_joint_windows(parent, root, config, attempts_dir, remote_dir=None, environment=None, progress=True):\n    """Secuencia prefijada, checkpoints sellados por intento; se detiene en UNKNOWN.\n\n    Un checkpoint desconocido se conserva también al reanudar. Ampliar su búsqueda\n    requiere una revisión explícita del presupuesto, nunca omitirlo o borrarlo.\n    """\n    if (config[\'protocol_version\'] != PROTOCOL_VERSION\n            or config[\'windows\'][\'joint_policy\'] != JOINT_WINDOW_POLICY):\n        raise ValueError(\'Protocolo conjunto incompatible.\')\n    attempts_dir = Path(attempts_dir)\n    remote_dir = Path(remote_dir) if remote_dir else None\n    spec = Traffic(**config[\'traffic\'])\n    scale = config[\'certification\'][\'time_scale\']\n    limit = config[\'windows\'][\'max_joint_candidates\']\n    if not isinstance(limit, int) or isinstance(limit, bool) or limit < 1:\n        raise ValueError(\'Presupuesto de candidatos inválido.\')\n    history = {\'policy\': JOINT_WINDOW_POLICY, \'max_joint_candidates\': limit, \'attempts\': []}\n    for index in range(limit):\n        if progress:\n            print(f\'Intento conjunto {index + 1}/{limit} (índice {index}); geometría y flota fijas.\', flush=True)\n        candidate, ledger = make_conditioned_candidate(parent, root, spec,\n            config[\'windows\'][\'max_draws_per_customer\'], scale, joint_attempt=index)\n        validate_sampling_ledger(candidate, ledger)\n        checkpoint = attempts_dir / f\'attempt{index:03d}\'\n        remote = remote_dir / checkpoint.name if remote_dir else None\n        if remote and remote.exists() and not checkpoint.exists():\n            sync_verified(remote, checkpoint)\n        if checkpoint.exists():\n            verify_bundle(checkpoint)\n            record = json.loads((checkpoint / \'record.json\').read_text())\n            if (record[\'candidate\'] != candidate.payload() or record[\'ledger\'] != ledger\n                    or record[\'joint_attempt\'] != index\n                    or json.loads((checkpoint / \'config.json\').read_text()) != config):\n                raise ValueError(\'Checkpoint distinto de la secuencia/configuración regenerada.\')\n            certificate, report = record[\'certificate\'], record[\'certification\']\n            if progress:\n                print(\'Intento recuperado:\', report[\'status\'], flush=True)\n        else:\n            certificate, report = certify_candidate(candidate, spec,\n                config[\'certification\'][\'seconds_per_strategy\'], scale, progress=progress)\n            record = {\'joint_attempt\': index, \'candidate\': candidate.payload(), \'ledger\': ledger,\n                      \'certificate\': certificate, \'certification\': report}\n            write_json(checkpoint / \'record.json\', record)\n            write_json(checkpoint / \'config.json\', config)\n            write_json(checkpoint / \'environment.json\', environment or {})\n            seal_bundle(checkpoint)\n        if remote:\n            sync_verified(checkpoint, remote)\n        history[\'attempts\'].append(record)\n        if certificate is not None:\n            validate_admission_history(candidate, history, spec, scale)\n            return candidate, ledger, certificate, history\n        if not proven_rejection(candidate, report, spec, scale):\n            raise RuntimeError(f\'n={parent.n}: SIN CERTIFICADO ({report["status"]}). \'\n                f\'No se remuestrea ni se autoriza el banco. Conservar y revisar: {checkpoint}/record.json\')\n        if progress:\n            print(\'INFACTIBILIDAD PROBADA:\', report[\'status\'],\n                  \'→ rechazo registrado; siguiente muestra conjunta oficial.\', flush=True)\n    raise RuntimeError(f\'n={parent.n}: agotados {limit} candidatos con pruebas negativas. \'\n                       f\'Banco no autorizado; revisar protocolo. Registro: {attempts_dir}\')\n\n\ndef acceptance_report(instances, certificates, ledgers, config, histories=None):\n    if config[\'protocol_version\'] != PROTOCOL_VERSION or config[\'windows\'][\'policy\'] != WINDOW_POLICY:\n        raise ValueError(\'Protocolo incompatible.\')\n    if config[\'role\'] != \'development_only\' or config[\'traffic\'][\'mode\'] != \'stochastic\':\n        raise ValueError(\'00 sólo certifica desarrollo bajo tráfico estocástico declarado.\')\n    if config[\'sizes\'] != [50, 100, 200, 300] or sorted(x.n for x in instances) != config[\'sizes\']:\n        raise ValueError(\'Se requieren los cuatro tamaños del piloto.\')\n    if (config.get(\'depot_index\') != 0 or config.get(\'service_time_minutes\') != 0\n            or config.get(\'global_start_time_minutes\') != 0 or config.get(\'instances_per_size\') != 1):\n        raise ValueError(\'Convenciones configuradas no implementadas por el evaluador de 00.\')\n    spec = Traffic(**config[\'traffic\'])\n    for x in instances:\n        if x.provenance.get(\'kind\') != \'official_conditioned_windows\':\n            raise ValueError(\'Ventanas proyectadas o de otro origen no admitidas en v4.\')\n        validate_sampling_ledger(x, ledgers[x.n])\n        if histories is None or x.n not in histories:\n            raise ValueError(\'Falta historial conjunto para aceptar el banco v4.\')\n        history = histories[x.n]\n        if (config[\'windows\'][\'joint_policy\'] != JOINT_WINDOW_POLICY\n                or history[\'max_joint_candidates\'] != config[\'windows\'][\'max_joint_candidates\']):\n            raise ValueError(\'Política o presupuesto conjunto inconsistente.\')\n        validate_admission_history(x, history, spec, config[\'certification\'][\'time_scale\'])\n        if history[\'attempts\'][-1][\'certificate\'] != certificates[x.n]:\n            raise ValueError(\'Certificado seleccionado distinto del historial.\')\n        if certificates[x.n].get(\'policy\') != WINDOW_POLICY:\n            raise ValueError(\'Certificado de otra política de generación.\')\n        kinds = np.asarray(x.provenance[\'customer_types\'])\n        widths = x.windows[:, 1] - x.windows[:, 0]\n        if (kinds.shape != (x.n + 1,) or kinds[0] != -1 or not np.isin(kinds[1:], [0, 1]).all()\n                or not np.isin(widths[1:], [60, 120]).all() or np.any(widths[kinds == 1] != 60)\n                or not np.array_equal(x.windows[0], [0, 1440]) or np.any(x.appear_times != 0)):\n            raise ValueError(\'Tipos, duraciones, depósito o apariciones fuera del alcance declarado.\')\n        audit = audit_instance(x, spec, certificates[x.n])\n        if audit[\'proven_infeasible_by_lower_bound\'] or not audit[\'feasibility_certified\']:\n            raise ValueError(\'No se cumplen las condiciones nominales del banco.\')\n    return {\'status\': \'ready_for_method_development\', \'protocol_version\': PROTOCOL_VERSION,\n            \'instance_hashes\': {str(x.n): x.sha for x in instances},\n            \'geometry_groups\': {str(x.n): x.provenance[\'geometry_group_sha256\'] for x in instances},\n            \'all_nominal_certificates_verified\': True,\n            \'all_windows_from_official_generator_conditioned\': True,\n            \'all_rejections_proven_infeasible\': True,\n            \'admission_history_hashes\': {str(x.n): digest(histories[x.n]) for x in instances},\n            \'ready_for_confirmatory_comparison\': False,\n            \'limitations\': [\'one_development_geometry_per_size\', \'conditioned_window_distribution\',\n                            \'fixed_known_demands_and_zero_service\', \'nonbinding_capacity_in_source\',\n                            \'traffic_is_dynamic_not_online_customer_revelation\',\n                            \'no_almost_sure_stochastic_feasibility_guarantee\',\n                            \'complete_recourse_cost_objective_pending_before_training\']}\n\n\ndef diagnostic_greedy(x, spec=Traffic()):\n    """Heurística de arranque, no baseline del artículo ni óptimo.\n\n    Ordena clientes por cierre; asigna a la ruta con menor tardanza proyectada,\n    incremento de distancia y llegada, en ese orden lexicográfico. Sin escenarios.\n    """\n    d = distances(x)\n    routes = [[] for _ in x.capacities]\n    times = np.zeros(len(routes)); loads = np.zeros(len(routes))\n    for j in sorted(range(1, x.n + 1), key=lambda k: (x.windows[k, 1], k)):\n        choices = []\n        for v, route in enumerate(routes):\n            if loads[v] + x.demands[j] > x.capacities[v] + 1e-9:\n                continue\n            i = route[-1] if route else 0\n            arrival = max(times[v] + expected_time(d[i, j], times[v], spec),\n                          x.windows[j, 0], x.appear_times[j])\n            late = max(0., arrival - x.windows[j, 1])\n            depot_late = max(0., arrival + expected_time(d[j, 0], arrival, spec) - x.windows[0, 1])\n            choices.append(((late + depot_late, d[i, j] + d[j, 0] - d[i, 0], arrival, v), v, arrival))\n        if not choices:\n            raise ValueError(f\'Heurística no pudo asignar cliente {j}. No prueba infactibilidad global.\')\n        _, v, arrival = min(choices)\n        routes[v].append(j); times[v] = arrival; loads[v] += x.demands[j]\n    validate_routes(x, routes)\n    return routes\n\n\ndef simulate(x, routes, scenario_seed, spec=Traffic(), policy=\'fixed_order\', trace=False):\n    """Evalúa sin filtrar incumplimientos. Recurso sólo usa reloj y medias conocidas.\n\n    causal_edd: si siguiente cliente ya se proyecta tarde, elige entre los pendientes\n    aún alcanzables el de cierre más próximo; no cambia asignación de vehículo.\n    El servicio tardío se registra y continúa, no se convierte en factibilidad.\n    """\n    validate_routes(x, routes)\n    if policy not in (\'fixed_order\', \'causal_edd\'):\n        raise ValueError(\'Política de recurso desconocida.\')\n    d = distances(x)\n    travel = waiting = late_sum = overtime = base_cost = 0.\n    late_nodes, changed, actual_routes, events = set(), 0, [], []\n    for v, route in enumerate(routes):\n        path = [0] + list(route) + [0]\n        base_cost += sum(d[a, b] / spec.velocity for a, b in zip(path, path[1:]))\n        pending, actual, i, t = list(route), [], 0, 0.\n        while pending:\n            j = pending[0]\n            if policy == \'causal_edd\':\n                forecast = max(t + expected_time(d[i, j], t, spec), x.windows[j, 0], x.appear_times[j])\n                if forecast > x.windows[j, 1]:\n                    possible = [k for k in pending if max(t + expected_time(d[i, k], t, spec),\n                                x.windows[k, 0], x.appear_times[k]) <= x.windows[k, 1]]\n                    if possible:\n                        chosen = min(possible, key=lambda k: (x.windows[k, 1], d[i, k], k))\n                        changed += int(chosen != j)\n                        j = chosen\n            duration = travel_time(d[i, j], t, i, j, x.instance_id, scenario_seed, spec)\n            arrival = t + duration\n            start = max(arrival, x.windows[j, 0], x.appear_times[j])\n            late = max(0., start - x.windows[j, 1])\n            if late > 1e-9:\n                late_nodes.add(j)\n            travel += duration; waiting += start - arrival; late_sum += late\n            if trace:\n                events.append({\'vehicle\': v, \'from\': i, \'to\': j, \'departure\': t,\n                               \'arrival\': arrival, \'service_start\': start, \'late\': late})\n            pending.remove(j); actual.append(j); i, t = j, start\n        if route:\n            duration = travel_time(d[i, 0], t, i, 0, x.instance_id, scenario_seed, spec)\n            travel += duration\n            overtime += max(0., t + duration - x.windows[0, 1])\n            if trace:\n                events.append({\'vehicle\': v, \'from\': i, \'to\': 0, \'departure\': t,\n                               \'arrival\': t + duration, \'service_start\': t + duration,\n                               \'late\': max(0., t + duration - x.windows[0, 1])})\n        actual_routes.append(actual)\n    result = {\'scenario_seed\': int(scenario_seed), \'travel_minutes\': travel,\n              \'base_minutes\': base_cost, \'net_execution_delta_minutes\': travel - base_cost,\n              \'waiting_minutes\': waiting, \'lateness_minutes\': late_sum,\n              \'depot_overtime_minutes\': overtime, \'late_customers\': len(late_nodes),\n              \'customer_violation_fraction\': len(late_nodes) / x.n,\n              \'strict_feasible\': len(late_nodes) == 0 and overtime <= 1e-9,\n              \'all_customers_served\': True, \'used_vehicles\': sum(bool(r) for r in routes),\n              \'recourse_reorders\': changed}\n    if trace:\n        result.update(events=events, realized_routes=actual_routes)\n    return result\n\n\ndef empirical_cvar(values, alpha=.95):\n    values = np.asarray(values, dtype=np.float64)\n    if values.ndim != 1 or len(values) == 0 or not np.isfinite(values).all() or not 0 <= alpha < 1:\n        raise ValueError(\'Muestra/alpha inválidos para CVaR.\')\n    z = float(np.sort(values)[max(0, math.ceil(alpha * len(values)) - 1)])\n    return z + float(np.maximum(values - z, 0).sum()) / (len(values) * (1 - alpha))\n\n\ndef evaluate(x, routes, seeds, spec=Traffic(), policy=\'fixed_order\', alpha=.95):\n    before = x.sha\n    started = time.perf_counter()\n    records = [simulate(x, routes, s, spec, policy) for s in seeds]\n    if not records or len(set(seeds)) != len(seeds):\n        raise ValueError(\'Escenarios vacíos o repetidos.\')\n    assert x.sha == before, \'La instancia fue alterada.\'\n    costs = np.array([r[\'travel_minutes\'] for r in records])\n    summary = {\'instance_id\': x.instance_id, \'instance_sha256\': before, \'policy\': policy,\n               \'scenarios\': len(records), \'alpha\': alpha,\n               \'mean_travel_minutes\': float(costs.mean()),\n               \'cvar_travel_minutes\': empirical_cvar(costs, alpha),\n               \'cost_variance_population\': float(costs.var(ddof=0)),\n               \'scenario_strict_feasibility\': float(np.mean([r[\'strict_feasible\'] for r in records])),\n               \'mean_customer_violation_fraction\': float(np.mean([r[\'customer_violation_fraction\'] for r in records])),\n               \'mean_lateness_minutes\': float(np.mean([r[\'lateness_minutes\'] for r in records])),\n               \'mean_depot_overtime_minutes\': float(np.mean([r[\'depot_overtime_minutes\'] for r in records])),\n               \'used_vehicles\': records[0][\'used_vehicles\'],\n               \'evaluation_seconds\': time.perf_counter() - started}\n    return summary, records\n\n\ndef environment_report():\n    versions = {}\n    for pkg in (\'numpy\', \'pandas\', \'matplotlib\', \'scipy\', \'scikit-learn\', \'pillow\', \'pyarrow\', \'huggingface-hub\', \'ortools\', \'torch\'):\n        try:\n            versions[pkg] = importlib.metadata.version(pkg)\n        except importlib.metadata.PackageNotFoundError:\n            versions[pkg] = None\n    resolved = {}\n    for d in importlib.metadata.distributions():\n        try:\n            name = d.metadata[\'Name\']\n        except Exception:  # dist-info residual de pip sin METADATA\n            continue\n        if name:\n            resolved.setdefault(name, d.version)  # la primera en sys.path es la importada\n    report = {\'resolved_packages\': resolved, \'python\': sys.version, \'platform\': platform.platform(),\n              \'cpu_count\': os.cpu_count(), \'versions\': versions}\n    if shutil.which(\'nvidia-smi\'):\n        result = subprocess.run([\'nvidia-smi\', \'--query-gpu=name,memory.total,driver_version\',\n                                 \'--format=csv,noheader\'], capture_output=True, text=True, timeout=15)\n        report[\'gpu\'] = result.stdout.strip()\n    else:\n        report[\'gpu\'] = None\n    return report\n\n\ndef bundle_files(directory):\n    """Archivos sellables; excluye bytecode que Python crea al importar bank/code."""\n    return [p for p in sorted(directory.rglob(\'*\')) if p.is_file() and p.name != \'CHECKSUMS.json\'\n            and \'__pycache__\' not in p.relative_to(directory).parts]\n\n\ndef seal_bundle(directory):\n    directory = Path(directory)\n    manifest = {str(p.relative_to(directory)): file_hash(p) for p in bundle_files(directory)}\n    write_json(directory / \'CHECKSUMS.json\', manifest)\n    return digest(manifest)\n\n\ndef verify_bundle(directory):\n    directory = Path(directory)\n    manifest = json.loads((directory / \'CHECKSUMS.json\').read_text())\n    for name, expected in manifest.items():\n        path = directory / name\n        if not path.is_file() or file_hash(path) != expected:\n            raise ValueError(f\'Archivo ausente/modificado: {name}\')\n    actual = {str(p.relative_to(directory)) for p in bundle_files(directory)}\n    if actual != set(manifest):\n        raise ValueError(\'Archivos extra en banco sellado.\')\n    return digest(manifest)\n\n\ndef load_bank(directory):\n    """Punto de entrada común para notebooks 01–06."""\n    directory = Path(directory)\n    verify_bundle(directory)\n    frozen_code = directory / \'code\' / \'svrplab_pilot.py\'\n    if not frozen_code.exists() or file_hash(frozen_code) != file_hash(__file__):\n        raise ValueError(\'Evaluador diferente al banco. Importe el módulo guardado en bank/code.\')\n    config = json.loads((directory / \'config.json\').read_text())\n    if config.get(\'protocol_version\') != PROTOCOL_VERSION:\n        raise ValueError(\'Versión del banco incompatible; no mezclar bancos v1, v2, v3 y v4.\')\n    instances = [freeze_instance(json.loads(p.read_text())) for p in sorted((directory / \'instances\').glob(\'*.json\'))]\n    if sorted(x.n for x in instances) != sorted(config[\'sizes\']):\n        raise ValueError(\'Tamaños del banco no coinciden con configuración.\')\n    certificates, ledgers, histories = {}, {}, {}\n    for x in instances:\n        parent = freeze_instance(json.loads((directory / f\'sources/reconstructed_v1_n{x.n:04d}.json\').read_text()))\n        if parent.sha != x.provenance[\'transformation\'][\'parent_instance_sha256\']:\n            raise ValueError(\'Fuente reconstruida distinta del padre declarado.\')\n        for field in (\'locations\', \'demands\', \'capacities\', \'appear_times\'):\n            if not np.array_equal(getattr(parent, field), getattr(x, field)):\n                raise ValueError(\'Derivado alteró geometría, demanda, flota o apariciones de su fuente.\')\n        if (not np.array_equal(parent.windows[:, 1] - parent.windows[:, 0], x.windows[:, 1] - x.windows[:, 0])\n                or not np.array_equal(parent.windows[0], x.windows[0])):\n            raise ValueError(\'Derivado alteró duración de ventanas o depósito.\')\n        certificate = json.loads((directory / f\'certificates/n{x.n:04d}.json\').read_text())\n        certificates[x.n] = certificate\n        ledgers[x.n] = json.loads((directory / f\'sources/sampling_ledger_n{x.n:04d}.json\').read_text())\n        histories[x.n] = json.loads((directory / f\'sources/admission_n{x.n:04d}.json\').read_text())\n    report = acceptance_report(instances, certificates, ledgers, config, histories)\n    stored = json.loads((directory / \'acceptance.json\').read_text())\n    if stored != report:\n        raise ValueError(\'Acta de aceptación distinta de la validación recalculada.\')\n    return instances\n\n\ndef sync_verified(source, destination):\n    """Copia comprobada, sin borrar origen ni sobrescribir un banco diferente."""\n    source, destination = Path(source), Path(destination)\n    expected = verify_bundle(source)\n    if destination.exists():\n        if verify_bundle(destination) != expected:\n            raise ValueError(f\'Ya existe otro contenido en {destination}; use otro ID.\')\n        return destination\n    staging = destination.with_name(destination.name + \'.partial\')\n    if staging.exists():\n        raise ValueError(f\'Copia interrumpida en {staging}; conservar e inspeccionar antes de reintentar.\')\n    staging.parent.mkdir(parents=True, exist_ok=True)\n    shutil.copytree(source, staging)\n    if verify_bundle(staging) != expected:\n        raise IOError(\'Copia no verificada; origen conservado.\')\n    staging.rename(destination)\n    return destination\n', 'final_experiment.py': '"""10: ejecución holdout independiente exclusivamente en Colab.\n\nEl verificador de exportaciones sólo usa stdlib y el evaluador auditado. Los\nentrenadores/solvers se importan dentro de las acciones explícitas. No se consulta\nningún certificado al construir una solución. Esta vía no altera methods-v2.\n"""\nfrom __future__ import annotations\n\nimport csv\nimport gzip\nimport json\nimport math\nimport shutil\nimport time\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nfrom . import pilot as p\n\nSCHEMA = \'final-experiment-v1\'\nEXPORT_SCHEMA = \'final-analysis-export-v1\'\nSPLIT = \'independent_train_validation_test_v1\'\n\n\ndef _colab():\n    try:\n        import google.colab  # noqa: F401\n    except ImportError as exc:\n        raise RuntimeError(\'Entrenamiento, evaluación y pruebas exclusivamente en Google Colab.\') from exc\n\n\ndef _read(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\n\ndef _now():\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef _sources():\n    return {path.name: p.file_hash(path) for path in sorted(Path(__file__).parent.glob(\'*.py\'))}\n\n\ndef _safe(root, relative):\n    root, relative = Path(root).resolve(), Path(relative)\n    if relative.is_absolute() or \'..\' in relative.parts or not relative.parts:\n        raise ValueError(\'Ruta relativa inválida en manifiesto.\')\n    target = root / relative\n    if not target.resolve().is_relative_to(root):\n        raise ValueError(\'Ruta fuera del artefacto autorizado.\')\n    return target\n\n\ndef _csv(path):\n    with Path(path).open(newline=\'\', encoding=\'utf-8\') as stream:\n        return list(csv.DictReader(stream))\n\n\ndef validate_config(config, design):\n    from . import hybrid as hy\n    from . import reinforcement as rl\n    from .research_design import scenario_precision_plan\n    if (config[\'schema_version\'] != SCHEMA or config[\'role\'] != \'confirmatory_holdout\'\n            or config[\'split_strategy\'] != SPLIT or config[\'execution_policy\'] != \'colab_only\'\n            or config[\'objective\'] != \'conservative_upper_travel_minutes_v1\'\n            or config[\'solver_seeds\'] != design[\'seed_ids\']\n            or config[\'search_seconds\'] != design[\'search_seconds\']\n            or config[\'policies\'] != [\'fixed_order\', \'causal_edd\']):\n        raise ValueError(\'Contrato final incompatible con el diseño sellado.\')\n    expected = {\'hbg_facs\', \'hbg_untrained\', \'hbg_no_pheromone\', \'pomo\', \'pomo_untrained\',\n                \'gurobi\', \'edd\', \'ortools_routing\', \'aco\', \'tabu\'}\n    if set(config[\'methods\']) != expected or config[\'methods\'][\'pomo\'] != config[\'methods\'][\'pomo_untrained\']:\n        raise ValueError(\'Inventario final incompleto o control POMO con búsqueda distinta.\')\n    hy.validate_inference_controls(config)\n    hy.validate_training_config(config[\'training\'][\'hbg_facs\'])\n    rl.validate_training_config(config[\'training\'][\'pomo\'])\n    for settings in config[\'training\'].values():\n        if settings[\'time_scale\'] != config[\'time_scale\'] or settings[\'objective\'] != config[\'objective\']:\n            raise ValueError(\'Entrenamiento e inferencia usan objetivos/escalas distintos.\')\n        if settings[\'training\'].get(\'geometry_sampling\') != \'shuffled_without_replacement\':\n            raise ValueError(\'El entrenamiento final requiere cobertura ordenada de geometrías.\')\n    scenarios = config[\'scenarios\']\n    precision = scenario_precision_plan(design)\n    required = max(math.ceil(math.log(2 / design[\'fr_error_probability\']) / (2 * design[\'fr_absolute_error\'] ** 2)),\n                   math.ceil(design[\'minimum_expected_tail_observations\'] / (1 - design[\'cvar_alpha\'])))\n    if (type(scenarios[\'evaluation_count\']) is not int or scenarios[\'evaluation_count\'] < required\n            or not 0 <= scenarios[\'trace_audit_first_scenarios\'] <= scenarios[\'evaluation_count\']\n            or scenarios[\'export\'] != \'scenario_metrics_csv_gzip_all_scenarios_with_fixed_trace_audit\'):\n        raise ValueError(\'Escenarios/exportación no cumplen precisión y auditoría prefijadas.\')\n    if config[\'training_geometry_minimum_observations\'] < 1:\n        raise ValueError(\'No se permite congelar entrenamiento sin exposición de geometrías.\')\n    return precision\n\n\ndef resource_plan(registry, config):\n    test = [row for row in registry if row[\'role\'] == \'test\']\n    attempts = len(test) * len(config[\'methods\']) * len(config[\'solver_seeds\'])\n    rows = attempts * len(config[\'policies\']) * config[\'scenarios\'][\'evaluation_count\']\n    training = sum(value[\'training\'][\'seconds_per_model\'] for value in config[\'training\'].values()) * len(config[\'solver_seeds\'])\n    hours = attempts * config[\'search_seconds\'] / 3600\n    limits = config[\'resource_limits\']\n    return {\'test_instances\': len(test), \'solver_attempts\': attempts,\n            \'planned_search_budget_hours\': hours, \'planned_training_budget_hours\': training / 3600,\n            \'scenario_metric_rows_upper_bound\': rows,\n            \'client_visit_event_rows_avoided\': sum(row[\'n_customers\'] for row in test) * len(config[\'methods\']) * len(config[\'solver_seeds\']) * len(config[\'policies\']) * config[\'scenarios\'][\'evaluation_count\'],\n            \'passes_declared_resource_limits\': hours <= limits[\'maximum_planned_search_hours\'] and training / 3600 <= limits[\'maximum_planned_training_hours\'] and rows <= limits[\'maximum_scenario_rows\'],\n            \'limits_are_compute_budgets_not_wall_time_guarantees\': True,\n            \'evaluation_io_validation_and_restart_overheads_are_additional\': True}\n\n\n\ndef recover_contract_validation(directory, remote, source_sha256):\n    """Recupera el acta original; sus duraciones/entorno no cambian al reanudar.\n\n    Un acta fallida con las mismas fuentes se conserva y detiene el flujo. Una\n    ausencia permite ejecutar pytest una vez y sellar su resultado en el caller.\n    """\n    from . import experiments as e\n    directory, remote = Path(directory), Path(remote)\n    if not e._recover(directory, remote):\n        return False\n    validation = _read(directory / \'validation.json\')\n    if validation.get(\'source_sha256\') != source_sha256:\n        raise ValueError(\'Acta Colab recuperada de otras fuentes; no sustituirla.\')\n    if validation.get(\'status\') != \'passed\' or validation.get(\'returncode\') != 0:\n        raise ValueError(\'La validación Colab previa falló; se conserva y no se repite bajo el mismo sello.\')\n    return True\n\ndef prepare_protocol(population_root, population_report_sha256, design, config, analysis_source,\n                     analysis_config, validation_bundle, work_root, drive_root):\n    """Verifica población y sella decisiones/código/R ANTES de entrenar o abrir test."""\n    _colab()\n    from .research_design import load_population\n    from . import experiments as e\n    validate_config(config, design)\n    instances, registry, manifest = load_population(population_root, population_report_sha256, design)\n    if manifest[\'evaluator_sha256\'] != p.file_hash(p.__file__):\n        raise ValueError(\'El evaluador activo no es el que certificó la población.\')\n    validation_bundle = Path(validation_bundle)\n    validation_sha = p.verify_bundle(validation_bundle)\n    validation = _read(validation_bundle / \'validation.json\')\n    sources = _sources()\n    if validation.get(\'status\') != \'passed\' or validation.get(\'source_sha256\') != sources:\n        raise ValueError(\'Faltan pruebas Colab aprobadas para estas mismas fuentes.\')\n    if not Path(analysis_source).is_file() or not Path(analysis_config).is_file():\n        raise ValueError(\'Faltan fuente R/configuración que deben congelarse antes del test.\')\n    plan = resource_plan(registry, config)\n    if not plan[\'passes_declared_resource_limits\']:\n        raise ValueError(\'El plan excede los recursos prefijados; revisar ANTES de exposición final.\')\n    core = {\'schema_version\': SCHEMA, \'role\': \'confirmatory_holdout\', \'config\': config, \'design\': design,\n            \'population_sha256\': manifest[\'population_sha256\'], \'registry_sha256\': p.digest(registry),\n            \'registry\': registry, \'population_manifest\': manifest, \'source_sha256\': sources,\n            \'evaluator_sha256\': p.file_hash(p.__file__), \'analysis_source_sha256\': p.file_hash(analysis_source),\n            \'analysis_r_sha256\': p.file_hash(analysis_source),\n            \'analysis_config_sha256\': p.file_hash(analysis_config), \'validation_bundle_sha256\': validation_sha,\n            \'resource_plan\': plan, \'scenario_seeds\': {row[\'instance_sha256\']:\n                [p.seed_for(\'final10-evaluation\', config[\'scenarios\'][\'master_seed\'], row[\'instance_sha256\'], i)\n                 for i in range(config[\'scenarios\'][\'evaluation_count\'])] for row in registry if row[\'role\'] == \'test\'}}\n    protocol_id = \'final10_\' + p.digest(core)[:20]\n    root, remote = Path(work_root) / manifest[\'population_sha256\'] / protocol_id, Path(drive_root) / manifest[\'population_sha256\'] / protocol_id\n    if e._recover(root / \'protocol\', remote / \'protocol\'):\n        existing = _read(root / \'protocol/protocol.json\')\n        if {key: value for key, value in existing.items() if key != \'sealed_at\'} != core:\n            raise ValueError(\'Protocolo existente distinto; no cambiar después de iniciarlo.\')\n    else:\n        e.write_json(root / \'protocol/protocol.json\', {**core, \'sealed_at\': _now()})\n        e.write_json(root / \'protocol/research_design.json\', design)\n        e.write_json(root / \'protocol/population.json\', _read(Path(population_root) / \'reports\' / population_report_sha256 / \'population.json\'))\n        shutil.copy2(analysis_source, root / \'protocol/confirmatory_results_08.R\')\n        shutil.copy2(analysis_config, root / \'protocol/analysis_08.json\')\n        e.write_json(root / \'protocol/environment.json\', p.environment_report())\n        e._commit(root / \'protocol\', remote / \'protocol\')\n    return root, remote, instances\n\n\ndef _protocol(root):\n    root = Path(root)\n    p.verify_bundle(root / \'protocol\')\n    protocol = _read(root / \'protocol/protocol.json\')\n    if (protocol[\'source_sha256\'] != _sources() or protocol[\'evaluator_sha256\'] != p.file_hash(p.__file__)\n            or p.file_hash(root / \'protocol/confirmatory_results_08.R\') != protocol[\'analysis_r_sha256\']\n            or p.file_hash(root / \'protocol/analysis_08.json\') != protocol[\'analysis_config_sha256\']):\n        raise ValueError(\'Código/evaluador cambió: no entrenar ni abrir test bajo este protocolo.\')\n    return protocol\n\n\ndef _holdout_fold(protocol):\n    registry = protocol[\'registry\']\n    return {\'fold_id\': \'independent_holdout\', \'split_strategy\': SPLIT,\n            \'split_registry_sha256\': p.digest(registry),\n            **{destination: sorted({row[\'geometry_group_sha256\'] for row in registry if row[\'role\'] == source})\n               for source, destination in [(\'train\', \'train_geometry_groups\'), (\'validation\', \'validation_geometry_groups\'), (\'test\', \'heldout_geometry_groups\')]}}\n\n\ndef train_and_freeze(root, remote, instances, device=\'cuda\'):\n    """Tres semillas por familia global; geometrías test nunca construidas por trainer."""\n    _colab()\n    from . import experiments as e, hybrid as hy, reinforcement as rl\n    import torch\n    root, remote = Path(root), Path(remote)\n    protocol = _protocol(root)\n    config, fold = protocol[\'config\'], _holdout_fold(protocol)\n    if sorted(x.sha for x in instances) != sorted(row[\'instance_sha256\'] for row in protocol[\'registry\']):\n        raise ValueError(\'Instancias diferentes de la población sellada.\')\n    if config[\'require_cuda\'] and (str(device) != \'cuda\' or not torch.cuda.is_available()):\n        raise RuntimeError(\'Entrenamiento final requiere GPU CUDA de Colab.\')\n    if e._recover(root / \'freeze\', remote / \'freeze\'):\n        recovered = _read(root / \'freeze/freeze.json\')\n        for row in recovered[\'models\']:\n            relative = Path(row[\'relative_directory\'])\n            if not e._recover(_safe(root, relative), _safe(remote, relative)):\n                raise ValueError(\'No se recuperó el modelo final desde Drive.\')\n            model_meta = relative.parent / \'meta\'\n            if not e._recover(_safe(root, model_meta), _safe(remote, model_meta)):\n                raise ValueError(\'No se recuperó la identidad del entrenamiento desde Drive.\')\n            checkpoint = Path(row[\'checkpoint_relative_path\'])\n            if checkpoint.parent != relative and not e._recover(_safe(root, checkpoint.parent), _safe(remote, checkpoint.parent)):\n                raise ValueError(\'Falta checkpoint diagnóstico rechazado en Drive.\')\n        _verify_freeze(root, protocol)\n        return root / \'freeze\'\n    models = []\n    spec = p.Traffic(**protocol[\'population_manifest\'][\'traffic\'])\n    for method, trainer in [(\'hbg_facs\', hy), (\'pomo\', rl)]:\n        for seed in config[\'solver_seeds\']:\n            relative = Path(\'training\') / method / f\'seed_{seed}\'\n            try:\n                final = trainer.train_fold(instances, spec, config[\'training\'][method], seed, fold,\n                    root / relative, remote / relative, device, split_strategy=SPLIT)\n            except Exception as exc:\n                failure = root / \'technical_failures\' / method / f\'seed_{seed}\'\n                if not (failure / \'CHECKSUMS.json\').exists():\n                    e.write_json(failure / \'failure.json\', {\'method\': method, \'seed\': seed,\n                        \'error_type\': type(exc).__name__, \'message\': str(exc), \'test_exposure_authorized\': False})\n                    e._commit(failure, remote / \'technical_failures\' / method / f\'seed_{seed}\')\n                raise\n            summary = _read(final / \'summary.json\')\n            checkpoint = rl.selected_checkpoint_path(final, summary)\n            models.append({\'method\': method, \'seed\': seed, \'relative_directory\': str(final.relative_to(root)),\n                           \'checkpoint_relative_path\': str(checkpoint.relative_to(root)),\n                           \'bundle_sha256\': p.verify_bundle(final), \'summary\': summary})\n    def visits_for(row):\n        summary = row[\'summary\']\n        return (summary[\'selected_checkpoint_training_geometry_visits\'] if summary[\'selection_status\'] == \'selected_complete_validation\'\n                else summary[\'training_geometry_visits\'])\n    coverage = all(set(visits_for(row)) == set(fold[\'train_geometry_groups\']) and\n                   all(count >= config[\'training_geometry_minimum_observations\'] for count in visits_for(row).values()) for row in models)\n    if not coverage:\n        failure = root / \'training_gate_failure\'\n        if not e._recover(failure, remote / \'training_gate_failure\'):\n            e.write_json(failure / \'failure.json\', {\'status\': \'training_geometry_coverage_failed\',\n                         \'test_opened\': False, \'models\': models})\n            e._commit(failure, remote / \'training_gate_failure\')\n        raise ValueError(\'No se observó cada geometría train: aumentar presupuesto sólo antes de abrir test mediante otro protocolo.\')\n    freeze = {\'schema_version\': \'final-model-freeze-v1\', \'frozen_at\': _now(),\n              \'protocol_sha256\': p.file_hash(root / \'protocol/protocol.json\'),\n              \'population_sha256\': protocol[\'population_sha256\'], \'registry_sha256\': protocol[\'registry_sha256\'],\n              \'source_sha256\': protocol[\'source_sha256\'], \'analysis_source_sha256\': protocol[\'analysis_source_sha256\'],\n              \'analysis_r_sha256\': protocol[\'analysis_r_sha256\'],\n              \'analysis_config_sha256\': protocol[\'analysis_config_sha256\'], \'models\': models,\n              \'all_training_geometries_observed\': coverage, \'heldout_in_training_or_selection\': False,\n              \'all_seeds_retained\': True, \'rejected_checkpoints_are_failures_not_models\': True}\n    e.write_json(root / \'freeze/freeze.json\', freeze)\n    e._commit(root / \'freeze\', remote / \'freeze\')\n    _verify_freeze(root, protocol)\n    return root / \'freeze\'\n\n\ndef _verify_freeze(root, protocol):\n    root = Path(root)\n    p.verify_bundle(root / \'freeze\')\n    freeze = _read(root / \'freeze/freeze.json\')\n    if (freeze[\'protocol_sha256\'] != p.file_hash(root / \'protocol/protocol.json\')\n            or freeze[\'population_sha256\'] != protocol[\'population_sha256\']\n            or freeze[\'source_sha256\'] != protocol[\'source_sha256\']):\n        raise ValueError(\'Freeze no corresponde al protocolo/población/código.\')\n    if (freeze[\'all_training_geometries_observed\'] is not True or freeze[\'heldout_in_training_or_selection\'] is not False\n            or freeze[\'analysis_r_sha256\'] != protocol[\'analysis_r_sha256\']\n            or freeze[\'analysis_config_sha256\'] != protocol[\'analysis_config_sha256\']\n            or datetime.fromisoformat(freeze[\'frozen_at\']) < datetime.fromisoformat(protocol[\'sealed_at\'])):\n        raise ValueError(\'Freeze no acredita cobertura, orden temporal o análisis prefijado.\')\n    expected = {(method, seed) for method in protocol[\'config\'][\'training\'] for seed in protocol[\'config\'][\'solver_seeds\']}\n    if {(row[\'method\'], row[\'seed\']) for row in freeze[\'models\']} != expected or len(freeze[\'models\']) != len(expected):\n        raise ValueError(\'Inventario de entrenamiento final incompleto o duplicado.\')\n    for row in freeze[\'models\']:\n        final = _safe(root, row[\'relative_directory\'])\n        if p.verify_bundle(final) != row[\'bundle_sha256\'] or _read(final / \'summary.json\') != row[\'summary\']:\n            raise ValueError(\'Resumen de modelo alterado.\')\n        if p.file_hash(_safe(root, row[\'checkpoint_relative_path\'])) != row[\'summary\'][\'model_checkpoint_sha256\']:\n            raise ValueError(\'Pesos/checkpoint diagnóstico alterados después de freeze.\')\n        summary = row[\'summary\']\n        diagnostics = summary.get(\'best_validation_diagnostics\') or {}\n        if summary[\'selection_status\'] not in (\'selected_complete_validation\', \'validation_gate_failed\'):\n            raise ValueError(\'Intento neuronal no tiene desenlace válido.\')\n        if summary[\'selection_status\'] == \'selected_complete_validation\' and not (diagnostics.get(\'checkpoint_eligible\')\n                and set(diagnostics.get(\'sizes_with_complete_geometry\', {})) == {str(size) for size in protocol[\'design\'][\'sizes\']}\n                and all(diagnostics[\'sizes_with_complete_geometry\'].values())):\n            raise ValueError(\'Modelo no cumple la puerta final por tamaños antes de abrir test.\')\n        p.verify_bundle(final.parent / \'meta\')\n        trained_identity = _read(final.parent / \'meta/identity.json\')\n        if trained_identity[\'config\'] != protocol[\'config\'][\'training\'][row[\'method\']] or trained_identity[\'split_strategy\'] != SPLIT:\n            raise ValueError(\'Se entrenó con otro protocolo/configuración final.\')\n        if row[\'summary\'][\'fold\'] != _holdout_fold(protocol) or row[\'summary\'][\'split_strategy\'] != SPLIT:\n            raise ValueError(\'Un checkpoint se entrenó con otra partición.\')\n        visits = (row[\'summary\'][\'selected_checkpoint_training_geometry_visits\'] if row[\'summary\'][\'selection_status\'] == \'selected_complete_validation\'\n                  else row[\'summary\'][\'training_geometry_visits\'])\n        if set(visits) != set(_holdout_fold(protocol)[\'train_geometry_groups\']) or any(value < protocol[\'config\'][\'training_geometry_minimum_observations\'] for value in visits.values()):\n            raise ValueError(\'Freeze no acredita cobertura completa de entrenamiento.\')\n    return freeze\n\n\ndef _make_solvers(root, protocol, freeze, device, gurobi_env):\n    from . import hybrid as hy, reinforcement as rl, hbg, pomo, baselines, metaheuristics, exact\n    from .attention import AttentionPolicy\n    from .supervised import _load_checkpoint, synchronize\n    import torch\n    models = {(row[\'method\'], row[\'seed\']): row for row in freeze[\'models\']}\n    config = protocol[\'config\']\n\n    def neural(family, untrained=False):\n        def solve(problem, seed, settings, output):\n            started = time.perf_counter()\n            row = models[(family, seed)]\n            summary = row[\'summary\']\n            if not untrained and summary[\'selection_status\'] == \'validation_gate_failed\':\n                return problem.result(None, \'validation_gate_failed\', time.perf_counter() - started, [],\n                    validation_gate_failed=True, heldout_inference_executed=False,\n                    checkpoint_artifact_kind=\'unselected_epoch_state_for_audit_only\')\n            prefix = \'hybrid-05\' if family == \'hbg_facs\' else \'reinforcement-04\'\n            initialization_seed = p.seed_for(prefix, summary[\'fold\'][\'fold_id\'], seed)\n            if untrained:\n                torch.manual_seed(initialization_seed)\n                torch.cuda.manual_seed_all(initialization_seed)\n            model = (hbg.HBGPolicy if family == \'hbg_facs\' else AttentionPolicy)(summary[\'model_config\']).to(device)\n            try:\n                if untrained and family == \'hbg_facs\' and hy.initialization_fingerprint(model) != summary[\'initialization_state_sha256\']:\n                    raise ValueError(\'Control sin entrenar usa inicialización distinta.\')\n                if not untrained:\n                    model.load_state_dict(_load_checkpoint(_safe(root, row[\'checkpoint_relative_path\'])))\n                model.eval(); synchronize(device)\n                loading = time.perf_counter() - started\n                remaining = max(0., settings[\'seconds\'] - loading)\n                if remaining <= 0:\n                    result = problem.result(None, \'time_limit_model_load\', loading, [])\n                else:\n                    call = hbg.solve_hbg if family == \'hbg_facs\' else pomo.solve_pomo\n                    result = call(problem, seed, {**settings, \'seconds\': remaining}, model, device)\n                synchronize(device)\n                result[\'solve_seconds\'] = time.perf_counter() - started\n                result[\'metadata\'].update(model_load_seconds=loading, learned=not untrained,\n                    initialization_seed=initialization_seed, selection_status=summary[\'selection_status\'],\n                    training_seconds=0. if untrained else summary[\'training_seconds\'],\n                    model_checkpoint_sha256=None if untrained else summary[\'model_checkpoint_sha256\'])\n                return result\n            finally:\n                del model\n                if str(device).startswith(\'cuda\'):\n                    torch.cuda.empty_cache()\n        return solve\n\n    return {\'hbg_facs\': neural(\'hbg_facs\'), \'hbg_no_pheromone\': neural(\'hbg_facs\'),\n            \'hbg_untrained\': neural(\'hbg_facs\', True), \'pomo\': neural(\'pomo\'),\n            \'pomo_untrained\': neural(\'pomo\', True),\n            \'gurobi\': lambda problem, seed, settings, output: exact.solve_exact(problem, seed, settings, env=gurobi_env, output_dir=output),\n            \'edd\': lambda problem, seed, settings, output: baselines.solve_edd(problem, seed, settings),\n            \'ortools_routing\': lambda problem, seed, settings, output: baselines.solve_ortools_routing(problem, seed, settings),\n            \'aco\': lambda problem, seed, settings, output: metaheuristics.solve_aco(problem, seed, settings),\n            \'tabu\': lambda problem, seed, settings, output: metaheuristics.solve_tabu(problem, seed, settings)}\n\n\ndef _claim_test(root, remote, protocol, freeze, exposure_root):\n    """Un reclamo por población: sólo reanuda la misma identidad completa."""\n    from . import experiments as e\n    root, remote, exposure_root = Path(root), Path(remote), Path(exposure_root)\n    core = {\'schema_version\': \'single-population-test-exposure-v1\', \'population_sha256\': protocol[\'population_sha256\'],\n            \'protocol_sha256\': p.file_hash(root / \'protocol/protocol.json\'),\n            \'freeze_sha256\': p.file_hash(root / \'freeze/freeze.json\'), \'release_id\': root.name,\n            \'analysis_source_sha256\': protocol[\'analysis_source_sha256\']}\n    claim = exposure_root / protocol[\'population_sha256\'] / \'exposure.json\'\n    claim.parent.mkdir(parents=True, exist_ok=True)\n    if claim.exists():\n        existing = _read(claim)\n        if {key: value for key, value in existing.items() if key != \'opened_at\'} != core:\n            raise ValueError(\'Esta población test ya fue expuesta con otra identidad; no permitir otro protocolo.\')\n    else:\n        existing = {**core, \'opened_at\': _now()}\n        # Exclusión a nivel del filesystem montado: una sesión concurrente no reemplaza la reserva.\n        with claim.open(\'x\', encoding=\'utf-8\') as stream:\n            json.dump(existing, stream, ensure_ascii=False, indent=2)\n    for group in _holdout_fold(protocol)[\'heldout_geometry_groups\']:\n        group_claim = exposure_root / \'geometries\' / f\'{group}.json\'\n        group_claim.parent.mkdir(parents=True, exist_ok=True)\n        if group_claim.exists():\n            if _read(group_claim) != existing:\n                raise ValueError(\'Alguna geometría test ya está reservada por otra exposición.\')\n        else:\n            with group_claim.open(\'x\', encoding=\'utf-8\') as stream:\n                json.dump(existing, stream, ensure_ascii=False, indent=2)\n    if datetime.fromisoformat(existing[\'opened_at\']) < datetime.fromisoformat(freeze[\'frozen_at\']):\n        raise ValueError(\'El test fue abierto antes del freeze de modelos.\')\n    if e._recover(root / \'test\', remote / \'test\'):\n        if _read(root / \'test/exposure.json\') != existing:\n            raise ValueError(\'Registro local/remoto de exposición distinto.\')\n    else:\n        e.write_json(root / \'test/exposure.json\', existing)\n        e._commit(root / \'test\', remote / \'test\')\n    return existing\n\n\ndef run_test(root, remote, instances, exposure_root, *, device=\'cuda\', gurobi_env=None, open_test=False):\n    """Exposición deliberada; cada ruta se sella antes de sus escenarios comunes."""\n    _colab()\n    if open_test is not True:\n        raise ValueError(\'Test cerrado. Revisar recursos y fijar OPEN_FINAL_TEST=True expresamente en Colab.\')\n    from . import experiments as e\n    from .search import SearchProblem\n    import torch\n    root, remote = Path(root), Path(remote)\n    protocol = _protocol(root)\n    freeze = _verify_freeze(root, protocol)\n    config = protocol[\'config\']\n    if gurobi_env is None:\n        raise ValueError(\'Falta entorno/licencia Gurobi verificado; detener antes de exponer test.\')\n    if config[\'require_cuda\'] and (str(device) != \'cuda\' or not torch.cuda.is_available()):\n        raise ValueError(\'Dispositivo final diferente del protocolo; detener antes de exponer test.\')\n    expected = {row[\'instance_sha256\'] for row in protocol[\'registry\']}\n    if {x.sha for x in instances} != expected:\n        raise ValueError(\'Población distinta del protocolo.\')\n    exposure = _claim_test(root, remote, protocol, freeze, exposure_root)\n    if e._recover(root / \'report\', remote / \'report\'):\n        # En un runtime limpio no se copian millones de registros: verificar el\n        # reporte remoto junto a sus bundles hermanos y devolver esa misma ruta.\n        remote_report = remote / \'report\'\n        verify_analysis_export(remote_report, p.file_hash(root / \'report/CHECKSUMS.json\'))\n        return remote_report\n    solvers = _make_solvers(root, protocol, freeze, device, gurobi_env)\n    runtime_environment = p.environment_report()\n    runtime_id = p.digest(runtime_environment)\n    spec = p.Traffic(**protocol[\'population_manifest\'][\'traffic\'])\n    nominal = p.Traffic(**{**protocol[\'population_manifest\'][\'traffic\'], \'mode\': \'nominal_mean\'})\n    by_sha = {x.sha: x for x in instances}\n    test_registry = [row for row in protocol[\'registry\'] if row[\'role\'] == \'test\']\n    rows, artifacts = [], {}\n    for entry in test_registry:\n        x = by_sha[entry[\'instance_sha256\']]\n        for method in config[\'methods\']:\n            for seed in config[\'solver_seeds\']:\n                relative = Path(\'attempts\') / x.sha / method / f\'seed_{seed}\'\n                job, remote_job = root / relative, remote / relative\n                context = {\'comparison_id\': root.name, \'final_release_id\': root.name,\n                    \'instance_id\': x.instance_id, \'instance_sha256\': x.sha, \'geometry_group_sha256\': entry[\'geometry_group_sha256\'],\n                    \'n_customers\': x.n, \'method\': method, \'solver_seed\': seed, \'artifact_status\': \'observed\'}\n                if not e._recover(job / \'solution\', remote_job / \'solution\'):\n                    (job / \'solution\').mkdir(parents=True, exist_ok=True)\n                    started = time.perf_counter()\n                    import resource\n                    torch.cuda.reset_peak_memory_stats()\n                    try:\n                        problem = SearchProblem(x, spec, config[\'time_scale\'])\n                        remaining = max(0., config[\'search_seconds\'] - (time.perf_counter() - started))\n                        settings = {**config[\'methods\'][method], \'seconds\': remaining}\n                        result = (solvers[method](problem, seed, settings, job / \'solution\') if remaining > 0\n                                  else problem.result(None, \'time_limit_preparation\', 0., []))\n                        if result.get(\'status\') in {\'license_pending\', \'solver_error\', \'not_available\', \'unavailable\'}:\n                            raise RuntimeError(\'Método no disponible: resolver infraestructura antes de continuar.\')\n                        result[\'search_total_seconds\'] = time.perf_counter() - started\n                        result[\'metadata\'].update(peak_cuda_allocated_bytes=int(torch.cuda.max_memory_allocated()),\n                            process_max_rss_kib=int(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss),\n                            cpu_memory_scope=\'process_lifetime_high_water_not_per_method_delta\',\n                            runtime_environment_sha256=runtime_id)\n                        structure = valid = False\n                        cost = None\n                        if result[\'routes\'] is not None:\n                            p.validate_routes(x, result[\'routes\']); structure = True\n                            violation, cost = problem.score(result[\'routes\'])\n                            valid = violation <= 1e-9\n                            if valid and not p.simulate(x, result[\'routes\'], 0, nominal)[\'strict_feasible\']:\n                                raise AssertionError(\'Solución conservadora incompatible con evaluador nominal.\')\n                        result.update(structural_valid=structure, conservative_feasible=valid,\n                                      search_objective_minutes=cost, context=context)\n                    except Exception as exc:\n                        failure = job / (\'technical_failure_\' + str(time.time_ns()))\n                        e.write_json(failure / \'failure.json\', {**context, \'error_type\': type(exc).__name__,\n                            \'message\': str(exc), \'counts_as_algorithm_failure\': False,\n                            \'resume_same_release_after_infrastructure_recovery\': True})\n                        e._commit(failure, remote_job / failure.name)\n                        raise RuntimeError(\'Fallo técnico registrado; reporte final bloqueado, no imputar derrota del método.\') from exc\n                    e.write_json(job / \'solution/environment.json\', runtime_environment)\n                    e.write_json(job / \'solution/solution.json\', result)\n                    e._commit(job / \'solution\', remote_job / \'solution\')\n                result = _read(job / \'solution/solution.json\')\n                if result[\'context\'] != context:\n                    raise ValueError(\'Solución recuperada de otro intento.\')\n                artifacts[str(relative / \'solution\')] = p.verify_bundle(job / \'solution\')\n                for policy in config[\'policies\']:\n                    if not e._recover(job / policy, remote_job / policy):\n                        summary = {**dict.fromkeys(e.SUMMARY_METRICS), **context, \'policy\': policy, \'solver_status\': result.get(\'status\'),\n                            \'structural_valid\': result[\'structural_valid\'], \'conservative_feasible\': result[\'conservative_feasible\'],\n                            \'search_objective_minutes\': result[\'search_objective_minutes\'],\n                            \'search_total_seconds\': result[\'search_total_seconds\'],\n                            \'evaluation_status\': \'no_solution\', \'scenario_strict_feasibility\': None,\n                            \'scenario_success_per_scheduled_attempt\': 0., \'model_selection_status\': result[\'metadata\'].get(\'selection_status\'),\n                            \'model_checkpoint_sha256\': result[\'metadata\'].get(\'model_checkpoint_sha256\'),\n                            \'relative_gap\': result[\'metadata\'].get(\'relative_gap\'), \'objective_bound\': result[\'metadata\'].get(\'best_bound\'), \'best_bound\': result[\'metadata\'].get(\'best_bound\'),\n                            \'surrogate_optimal\': result[\'metadata\'].get(\'surrogate_optimal\'),\n                            \'peak_cuda_allocated_bytes\': result[\'metadata\'].get(\'peak_cuda_allocated_bytes\'),\n                            \'process_max_rss_kib\': result[\'metadata\'].get(\'process_max_rss_kib\'),\n                            \'inference_environment_sha256\': result[\'metadata\'].get(\'runtime_environment_sha256\'),\n                            \'evaluation_environment_sha256\': runtime_id}\n                        if result[\'routes\'] is not None and result[\'structural_valid\']:\n                            seeds = protocol[\'scenario_seeds\'][x.sha]\n                            metrics, records = p.evaluate(x, result[\'routes\'], seeds, spec, policy, protocol[\'design\'][\'cvar_alpha\'])\n                            records_by_seed = {int(record[\'scenario_seed\']): record for record in records}\n                            if set(records_by_seed) != set(seeds):\n                                raise ValueError(\'Escenarios incompletos/repetidos en el evaluador común.\')\n                            (job / policy).mkdir(parents=True, exist_ok=True)\n                            with gzip.open(job / policy / \'scenario_metrics.csv.gz\', \'wt\', encoding=\'utf-8\', newline=\'\') as stream:\n                                writer = csv.DictWriter(stream, fieldnames=list(records[0])); writer.writeheader(); writer.writerows(records)\n                            audits = []\n                            trace_started = time.perf_counter()\n                            for scenario in seeds[:config[\'scenarios\'][\'trace_audit_first_scenarios\']]:\n                                trace = p.simulate(x, result[\'routes\'], scenario, spec, policy, trace=True)\n                                e.validate_trace(x, result[\'routes\'], trace, records_by_seed[scenario])\n                                audits.append(trace)\n                            e.write_json(job / policy / \'trace_audit.json\', audits)\n                            summary.update(metrics, evaluation_status=\'validated\',\n                                std_travel_minutes=math.sqrt(metrics[\'cost_variance_population\']),\n                                trace_validation_seconds=time.perf_counter() - trace_started,\n                                scenario_success_per_scheduled_attempt=metrics[\'scenario_strict_feasibility\'],\n                                mean_waiting_minutes=sum(row[\'waiting_minutes\'] for row in records) / len(records),\n                                mean_recourse_reorders=sum(row[\'recourse_reorders\'] for row in records) / len(records),\n                                mean_net_execution_delta_minutes=sum(row[\'net_execution_delta_minutes\'] for row in records) / len(records))\n                        e.write_json(job / policy / \'environment.json\', runtime_environment)\n                        e.write_json(job / policy / \'summary.json\', summary)\n                        e._commit(job / policy, remote_job / policy)\n                    rows.append(_read(job / policy / \'summary.json\'))\n                    artifacts[str(relative / policy)] = p.verify_bundle(job / policy)\n                print(x.instance_id, method, seed, result.get(\'status\'), flush=True)\n    return _export(root, remote, protocol, freeze, exposure, rows, artifacts)\n\n\ndef _export(root, remote, protocol, freeze, exposure, rows, artifacts):\n    from . import experiments as e\n    config = protocol[\'config\']\n    registry = [row for row in protocol[\'registry\'] if row[\'role\'] == \'test\']\n    expected = {(row[\'instance_sha256\'], method, seed, policy) for row in registry\n                for method in config[\'methods\'] for seed in config[\'solver_seeds\'] for policy in config[\'policies\']}\n    observed = {(row[\'instance_sha256\'], row[\'method\'], row[\'solver_seed\'], row[\'policy\']) for row in rows}\n    if observed != expected or len(rows) != len(expected):\n        raise ValueError(\'No exportar análisis con intentos ausentes o duplicados.\')\n    report = root / \'report\'\n    runtimes = {}\n    for relative in artifacts:\n        environment = _read(root / relative / \'environment.json\')\n        runtimes[p.digest(environment)] = environment\n    e.write_json(report / \'runtime_environments.json\', runtimes)\n    e.write_csv(report / \'runtime_summary.csv\', [{\'environment_sha256\': sha, \'gpu\': env.get(\'gpu\'),\n        \'cpu_count\': env.get(\'cpu_count\'), \'platform\': env.get(\'platform\'), \'python\': env.get(\'python\')}\n        for sha, env in sorted(runtimes.items())])\n    e.write_csv(report / \'attempts.csv\', rows)\n    e.write_csv(report / \'registry.csv\', registry)\n    e.write_json(report / \'research_design.json\', protocol[\'design\'])\n    test = {**exposure, \'completed_at\': _now(), \'artifact_bundle_sha256\': artifacts,\n            \'expected_policy_rows\': len(expected), \'observed_policy_rows\': len(rows),\n            \'complete_inventory\': True, \'result_replacement_allowed\': False}\n    source_objects = {\'population\': _read(root / \'protocol/population.json\'), \'protocol\': protocol,\n                      \'freeze\': freeze, \'test\': test}\n    source_hashes = {}\n    for name, obj in source_objects.items():\n        path = report / \'source_manifests\' / f\'{name}.json\'\n        e.write_json(path, obj); source_hashes[name] = p.file_hash(path)\n    release = {\'schema_version\': \'final-release-v1\', \'role\': \'confirmatory_holdout\', \'final_release_id\': root.name,\n        \'comparison_id\': root.name, \'population_sha256\': protocol[\'population_sha256\'],\n        \'protocol_sha256\': source_hashes[\'protocol\'], \'model_freeze_sha256\': source_hashes[\'freeze\'],\n        \'test_exposure_sha256\': p.file_hash(root / \'test/exposure.json\'),\n        \'model_freeze_before_test\': datetime.fromisoformat(freeze[\'frozen_at\']) <= datetime.fromisoformat(exposure[\'opened_at\']),\n        \'independent_geometry_split_verified\': True, \'test_evaluation_once\': True,\n        \'test_once_scope\': \'one_logged_exposure_identity_per_population_same_identity_resumption_only\',\n        \'analysis_source_sha256\': protocol[\'analysis_source_sha256\'], \'analysis_r_sha256\': protocol[\'analysis_r_sha256\'],\n        \'analysis_config_sha256\': protocol[\'analysis_config_sha256\'], \'complete_attempt_inventory\': True,\n        \'source_manifest_sha256\': source_hashes, \'distinct_runtime_environments\': len(runtimes),\n        \'timing_requires_hardware_stratification\': len(runtimes) > 1}\n    e.write_json(report / \'final_release.json\', release)\n    contract = {\'schema_version\': EXPORT_SCHEMA, \'role\': \'confirmatory_holdout\', \'analysis_mode\': \'confirmatory\',\n        \'unit\': \'geometry_group_sha256\', \'final_release_id\': root.name, \'comparison_id\': root.name,\n        \'expected_methods\': list(config[\'methods\']), \'solver_seeds\': config[\'solver_seeds\'], \'policies\': config[\'policies\'],\n        \'source_manifest_sha256\': source_hashes, \'analysis_source_sha256\': protocol[\'analysis_source_sha256\'],\n        \'analysis_r_sha256\': protocol[\'analysis_r_sha256\'],\n        \'analysis_config_sha256\': protocol[\'analysis_config_sha256\'],\n        \'all_failures_observed_missing_never_imputed\': True, \'events_scope\': config[\'scenarios\'][\'export\']}\n    e.write_json(report / \'analysis_export_contract.json\', contract)\n    e._commit(report, remote / \'report\')\n    verify_analysis_export(report, p.file_hash(report / \'CHECKSUMS.json\'))\n    print(\'EXPORT FINAL:\', remote / \'report\', \'\\nCHECKSUMS_SHA256:\', p.file_hash(report / \'CHECKSUMS.json\'), flush=True)\n    return report\n\n\ndef verify_analysis_export(directory, expected_manifest_sha256):\n    """Sólo lectura: verifica cadena y matriz completa antes de análisis R.\n\n    expected_manifest_sha256 es SHA-256 de los bytes CHECKSUMS.json copiado de 10,\n    no el hash canónico que retorna verify_bundle. No ejecuta fuentes desde Drive.\n    """\n    directory = Path(directory)\n    if p.file_hash(directory / \'CHECKSUMS.json\') != expected_manifest_sha256:\n        raise ValueError(\'Hash externo del reporte final distinto.\')\n    manifest = _read(directory / \'CHECKSUMS.json\')\n    for name in manifest:\n        _safe(directory, name)\n    p.verify_bundle(directory)\n    contract = _read(directory / \'analysis_export_contract.json\')\n    if (contract.get(\'schema_version\') != EXPORT_SCHEMA or contract.get(\'role\') != \'confirmatory_holdout\'\n            or contract.get(\'analysis_mode\') != \'confirmatory\' or contract.get(\'unit\') != \'geometry_group_sha256\'):\n        raise ValueError(\'Exportación no confirmatoria o de otro contrato.\')\n    chain = {}\n    for name in [\'population\', \'protocol\', \'freeze\', \'test\']:\n        path = directory / \'source_manifests\' / f\'{name}.json\'\n        if p.file_hash(path) != contract[\'source_manifest_sha256\'][name]:\n            raise ValueError(\'Cadena de fuentes alterada.\')\n        chain[name] = _read(path)\n    population, protocol, freeze, test = (chain[name] for name in [\'population\', \'protocol\', \'freeze\', \'test\'])\n    release = _read(directory / \'final_release.json\')\n    if (p.digest({key: value for key, value in population.items() if key != \'population_sha256\'}) != population[\'population_sha256\']\n            or population[\'status\'] != \'nominal_population_accepted\'\n            or protocol[\'population_sha256\'] != population[\'population_sha256\']\n            or freeze[\'population_sha256\'] != population[\'population_sha256\']\n            or test[\'population_sha256\'] != population[\'population_sha256\']\n            or release[\'population_sha256\'] != population[\'population_sha256\']):\n        raise ValueError(\'Población no aceptada o cadena de población distinta.\')\n    if (freeze[\'protocol_sha256\'] != contract[\'source_manifest_sha256\'][\'protocol\']\n            or test[\'protocol_sha256\'] != freeze[\'protocol_sha256\']\n            or test[\'freeze_sha256\'] != contract[\'source_manifest_sha256\'][\'freeze\']\n            or release[\'protocol_sha256\'] != freeze[\'protocol_sha256\']\n            or release[\'model_freeze_sha256\'] != test[\'freeze_sha256\']\n            or release[\'source_manifest_sha256\'] != contract[\'source_manifest_sha256\']):\n        raise ValueError(\'Protocolo, freeze y exposición no forman la misma cadena.\')\n    if not datetime.fromisoformat(protocol[\'sealed_at\']) <= datetime.fromisoformat(freeze[\'frozen_at\']) <= datetime.fromisoformat(test[\'opened_at\']) <= datetime.fromisoformat(test[\'completed_at\']):\n        raise ValueError(\'El orden sellado->freeze->test->completo es inválido.\')\n    if (contract[\'analysis_source_sha256\'] != protocol[\'analysis_source_sha256\']\n            or contract[\'analysis_config_sha256\'] != protocol[\'analysis_config_sha256\']\n            or freeze[\'analysis_source_sha256\'] != protocol[\'analysis_source_sha256\']\n            or release[\'analysis_source_sha256\'] != protocol[\'analysis_source_sha256\']):\n        raise ValueError(\'Código R/configuración cambiaron después de congelar el protocolo.\')\n    if (contract[\'expected_methods\'] != list(protocol[\'config\'][\'methods\'])\n            or contract[\'solver_seeds\'] != protocol[\'config\'][\'solver_seeds\']\n            or contract[\'policies\'] != protocol[\'config\'][\'policies\']\n            or contract[\'analysis_r_sha256\'] != protocol[\'analysis_r_sha256\']\n            or freeze[\'analysis_r_sha256\'] != protocol[\'analysis_r_sha256\']\n            or freeze[\'all_training_geometries_observed\'] is not True\n            or freeze[\'heldout_in_training_or_selection\'] is not False\n            or freeze[\'source_sha256\'] != protocol[\'source_sha256\']):\n        raise ValueError(\'Métodos, R, fuentes o comprobaciones de freeze difieren del protocolo.\')\n    registry = protocol[\'registry\']\n    if p.digest(registry) != protocol[\'registry_sha256\']:\n        raise ValueError(\'Registro de geometrías alterado.\')\n    role_groups = {role: {row[\'geometry_group_sha256\'] for row in registry if row[\'role\'] == role}\n                   for role in [\'train\', \'validation\', \'test\']}\n    excluded = set(protocol[\'population_manifest\'][\'excluded_development_groups\'])\n    if (any(not values for values in role_groups.values()) or role_groups[\'train\'] & role_groups[\'validation\']\n            or role_groups[\'train\'] & role_groups[\'test\'] or role_groups[\'validation\'] & role_groups[\'test\']\n            or set.union(*role_groups.values()) & excluded):\n        raise ValueError(\'La partición no es independiente del desarrollo.\')\n    fold = _holdout_fold(protocol)\n    expected_models = {(name, seed) for name in protocol[\'config\'][\'training\'] for seed in contract[\'solver_seeds\']}\n    if {(row[\'method\'], row[\'seed\']) for row in freeze[\'models\']} != expected_models or len(freeze[\'models\']) != len(expected_models):\n        raise ValueError(\'Faltan semillas/modelos en freeze.\')\n    for row in freeze[\'models\']:\n        summary = row[\'summary\']\n        if summary[\'fold\'] != fold or summary[\'heldout_used_for_training_or_selection\'] is not False:\n            raise ValueError(\'Modelo con fuga de test o partición distinta.\')\n        if summary[\'selection_status\'] not in [\'selected_complete_validation\', \'validation_gate_failed\']:\n            raise ValueError(\'Intento neuronal sin desenlace auditable.\')\n        visits = (summary[\'selected_checkpoint_training_geometry_visits\'] if summary[\'selection_status\'] == \'selected_complete_validation\'\n                  else summary[\'training_geometry_visits\'])\n        if set(visits) != set(fold[\'train_geometry_groups\']) or any(value < protocol[\'config\'][\'training_geometry_minimum_observations\'] for value in visits.values()):\n            raise ValueError(\'Falta cobertura de entrenamiento de todas las geometrías.\')\n        diagnostics = summary.get(\'best_validation_diagnostics\') or {}\n        if summary[\'selection_status\'] == \'selected_complete_validation\' and not (diagnostics.get(\'checkpoint_eligible\') and set(diagnostics.get(\'sizes_with_complete_geometry\', {})) == {str(size) for size in protocol[\'design\'][\'sizes\']} and all(diagnostics[\'sizes_with_complete_geometry\'].values())):\n            raise ValueError(\'Se congeló un modelo que no pasó completitud.\')\n    design = _read(directory / \'research_design.json\')\n    if design != protocol[\'design\'] or contract[\'solver_seeds\'] != design[\'seed_ids\']:\n        raise ValueError(\'Diseño/R no corresponden al protocolo congelado.\')\n    test_registry = [row for row in registry if row[\'role\'] == \'test\']\n    exported_registry = _csv(directory / \'registry.csv\')\n    if {row[\'instance_sha256\'] for row in exported_registry} != {row[\'instance_sha256\'] for row in test_registry} or len(exported_registry) != len(test_registry):\n        raise ValueError(\'Registro test exportado incompleto o duplicado.\')\n    by_export_sha = {row[\'instance_sha256\']: row for row in exported_registry}\n    for row in test_registry:\n        if by_export_sha[row[\'instance_sha256\']] != {key: str(value) for key, value in row.items()}:\n            raise ValueError(\'Registro exportado no conserva todos los campos del protocolo.\')\n    expected = {(row[\'instance_sha256\'], method, str(seed), policy) for row in test_registry\n                for method in contract[\'expected_methods\'] for seed in contract[\'solver_seeds\'] for policy in contract[\'policies\']}\n    attempts = _csv(directory / \'attempts.csv\')\n    observed = {(row[\'instance_sha256\'], row[\'method\'], row[\'solver_seed\'], row[\'policy\']) for row in attempts}\n    if observed != expected or len(attempts) != len(expected) or test[\'observed_policy_rows\'] != len(expected):\n        raise ValueError(\'Matriz de intentos incompleta o repetida; no imputar como fallos.\')\n    by_sha = {row[\'instance_sha256\']: row for row in test_registry}\n    for row in attempts:\n        origin = by_sha[row[\'instance_sha256\']]\n        if (row[\'artifact_status\'] != \'observed\' or row[\'geometry_group_sha256\'] != origin[\'geometry_group_sha256\']\n                or int(row[\'n_customers\']) != origin[\'n_customers\']):\n            raise ValueError(\'Identidad de intento incorrecta.\')\n    expected_artifacts = {str(Path(\'attempts\') / row[\'instance_sha256\'] / method / f\'seed_{seed}\' / part)\n                          for row in test_registry for method in contract[\'expected_methods\']\n                          for seed in contract[\'solver_seeds\'] for part in [\'solution\', *contract[\'policies\']]}\n    if set(test[\'artifact_bundle_sha256\']) != expected_artifacts:\n        raise ValueError(\'Inventario de soluciones/evaluaciones incompleto.\')\n    for relative, expected_hash in test[\'artifact_bundle_sha256\'].items():\n        artifact = _safe(directory.parent, relative)\n        if p.verify_bundle(artifact) != expected_hash:\n            raise ValueError(\'Solución o evaluación cambiada después de exportar.\')\n    for row in attempts:\n        relative = Path(\'attempts\') / row[\'instance_sha256\'] / row[\'method\'] / f\'seed_{row["solver_seed"]}\'\n        persisted = _read(_safe(directory.parent, relative / row[\'policy\'] / \'summary.json\'))\n        expected_row = {name: \'\' if value is None else str(value) for name, value in persisted.items()}\n        expected_row = {name: expected_row.get(name, \'\') for name in row}\n        if row != expected_row:\n            raise ValueError(\'CSV final difiere del resumen por intento sellado.\')\n        solution = _read(_safe(directory.parent, relative / \'solution\' / \'solution.json\'))\n        for record in (persisted, solution):\n            if (type(record.get(\'structural_valid\')) is not bool or type(record.get(\'conservative_feasible\')) is not bool\n                    or record[\'conservative_feasible\'] and not record[\'structural_valid\']):\n                raise ValueError(\'Flags de validez nulos, no booleanos o contradictorios.\')\n        if (persisted.get(\'evaluation_status\') not in (\'validated\', \'no_solution\')\n                or not isinstance(persisted.get(\'solver_status\'), str) or not persisted[\'solver_status\'].strip()\n                or persisted.get(\'artifact_status\') != \'observed\'\n                or persisted[\'solver_status\'] != solution.get(\'status\')):\n            raise ValueError(\'Estado de solver/evaluación ausente o no admitido.\')\n        if persisted[\'structural_valid\'] != solution[\'structural_valid\'] or persisted[\'conservative_feasible\'] != solution[\'conservative_feasible\']:\n            raise ValueError(\'La factibilidad exportada difiere de la solución congelada.\')\n        if solution[\'routes\'] is None and (persisted[\'evaluation_status\'] != \'no_solution\' or persisted[\'scenario_success_per_scheduled_attempt\'] != 0.):\n            raise ValueError(\'Intento sin rutas presentado como evaluado.\')\n    return {**contract, \'registry\': test_registry, \'design\': design, \'source_chain\': chain}\n'}
FINAL_R_SOURCE = '# Confirmación 08 sobre exporte final10 íntegro; ejecución sólo en Colab.\nargs <- commandArgs(trailingOnly = TRUE)\nif (Sys.getenv("SVRPLAB_COLAB_ANALYSIS") != "1" || length(args) != 2L ||\n    !file.exists("/content/.svrplab_final08_verified")) stop("Usar notebook 08 en Colab con verificación final10.")\ninput <- normalizePath(args[[1]], mustWork = TRUE); output <- args[[2]]\nif (dir.exists(output)) stop("No sobrescribir evidencia de análisis.")\ndir.create(output, recursive = TRUE)\noptions(stringsAsFactors = FALSE, warn = 2, OutDec = "."); Sys.setenv(TZ = "UTC")\nread <- function(name) read.csv(file.path(input, name), colClasses = "character", check.names = FALSE,\n                               na.strings = "", fileEncoding = "UTF-8")\nwrite <- function(x, name) write.csv(x, file.path(output, name), row.names = FALSE, na = "", fileEncoding = "UTF-8")\nneed <- function(x, fields) if (!all(fields %in% names(x))) stop(paste("Columnas faltantes:", paste(setdiff(fields, names(x)), collapse = ",")))\nnum <- function(x) {\n  value <- suppressWarnings(as.numeric(x))\n  if (any(!is.na(x) & !is.finite(value))) stop("Métrica no finita.")\n  value\n}\ntruth <- function(x) {\n  if (any(!is.na(x) & !x %in% c("True", "False", "true", "false"))) stop("Booleano inválido.")\n  !is.na(x) & x %in% c("True", "true")\n}\nkey <- function(x, fields) do.call(paste, c(x[fields], sep = "|"))\nmean_available <- function(x) if (all(is.na(x))) NA_real_ else mean(x[!is.na(x)])\nbridge <- read("analysis_bridge.csv"); meta <- setNames(bridge$value, bridge$key)\nif (meta[["schema_version"]] != "final-analysis-export-v1" || meta[["role"]] != "confirmatory_holdout" ||\n    meta[["analysis_mode"]] != "confirmatory" || meta[["primary_inference"]] != "stratified_geometry_weighted_hoeffding_mean_bound") stop("Contrato final distinto del análisis preespecificado.")\nmethods <- read("expected_methods.csv")$method; seeds <- read("expected_seeds.csv")$solver_seed\npolicies <- read("expected_policies.csv")$policy; sizes <- read("expected_sizes.csv")$n_customers\ncontrasts <- read("primary_contrasts.csv")\nif (nrow(contrasts) != 2L || !identical(contrasts$method, c("hbg_facs", "hbg_facs")) ||\n    !setequal(contrasts$reference_method, c("hbg_untrained", "hbg_no_pheromone"))) stop("Familia coprimaria alterada.")\nalpha <- num(meta[["familywise_alpha"]]); tie_abs <- num(meta[["cost_tie_absolute_minutes"]]); tie_rel <- num(meta[["cost_tie_relative"]])\nif (!identical(alpha, .05) || tie_abs < 0 || tie_rel < 0) stop("Parámetros estadísticos inválidos.")\nhashes <- read("input_file_hashes.csv")\nverify <- function() {\n  for (i in seq_len(nrow(hashes))) {\n    actual <- system2("sha256sum", shQuote(file.path(input, hashes$relative_path[[i]])), stdout = TRUE)\n    if (length(actual) != 1L || substr(actual, 1, 64) != hashes$sha256[[i]]) stop("Fuente final cambió.")\n  }\n}\nverify()\nregistry <- read("registry.csv")\nneed(registry, c("instance_sha256", "geometry_group_sha256", "n_customers"))\nif (anyNA(registry[c("instance_sha256", "geometry_group_sha256", "n_customers")]) || anyDuplicated(registry$instance_sha256) || anyDuplicated(registry$geometry_group_sha256)) stop("Test debe contener una instancia por geometría independiente.")\nif (!setequal(registry$n_customers, sizes)) stop("Estratos de tamaños incompletos.")\nquota <- num(meta[["test_geometries_per_size"]])\nif (any(table(factor(registry$n_customers, levels = sizes)) != quota)) stop("Cuota final diferente del sello.")\nattempts <- read("attempts.csv")\nkeys <- c("instance_sha256", "policy", "method", "solver_seed")\nneed(attempts, c(keys, "geometry_group_sha256", "n_customers", "evaluation_status", "solver_status",\n  "structural_valid", "conservative_feasible", "search_objective_minutes", "scenario_strict_feasibility",\n  "mean_travel_minutes", "cvar_travel_minutes", "mean_waiting_minutes", "mean_recourse_reorders", "search_total_seconds",\n  "artifact_status", "scenario_success_per_scheduled_attempt"))\n# Ausencia de un indicador de validez es un error de datos, nunca FALSE.\nrequired_status <- c("artifact_status", "solver_status", "evaluation_status", "structural_valid", "conservative_feasible")\nif (anyNA(attempts[required_status]) || any(attempts$artifact_status != "observed")) stop("Artefacto/estado/flags ausentes o no observados: detener análisis.")\nif (any(!attempts$evaluation_status %in% c("validated", "no_solution"))) stop("Estado de evaluación no admitido.")\nstructural <- truth(attempts$structural_valid)\nconservative <- truth(attempts$conservative_feasible)\nevaluated <- attempts$evaluation_status == "validated"\nif (any(conservative & !structural) || any(evaluated & !structural) || any(!evaluated & structural)) stop("Incoherencia entre evaluación, estructura y factibilidad conservadora.")\nif (any(attempts$solver_status %in% c("solver_error", "evaluation_error", "environment_error", "license_pending", "not_available", "unavailable"))) stop("Error técnico/disponibilidad: no convertir en fallo estadístico del método.")\nexpected <- expand.grid(instance_sha256 = registry$instance_sha256, policy = policies,\n                        method = methods, solver_seed = seeds, stringsAsFactors = FALSE)\nif (anyNA(attempts[keys]) || anyDuplicated(key(attempts, keys)) ||\n    !setequal(key(attempts, keys), key(expected, keys))) stop("Faltan/sobran artefactos: no recodificar como fallo.")\nreg <- registry[match(attempts$instance_sha256, registry$instance_sha256), ]\nfor (field in c("geometry_group_sha256", "n_customers")) if (any(attempts[[field]] != reg[[field]])) stop("Identidad final alterada.")\nfor (field in c("search_objective_minutes", "scenario_strict_feasibility", "mean_travel_minutes", "cvar_travel_minutes",\n                "mean_waiting_minutes", "mean_recourse_reorders", "search_total_seconds", "scenario_success_per_scheduled_attempt")) attempts[[field]] <- num(attempts[[field]])\nif (any(evaluated & is.na(attempts$scenario_strict_feasibility)) ||\n    any(!evaluated & (!is.na(attempts$scenario_strict_feasibility) | !is.na(attempts$search_objective_minutes))) ||\n    anyNA(attempts$scenario_success_per_scheduled_attempt)) stop("Estado de evaluación incompatible con mediciones registradas.")\nexpected_success <- ifelse(evaluated, attempts$scenario_strict_feasibility, 0)\nif (any(abs(attempts$scenario_success_per_scheduled_attempt - expected_success) > 1e-12)) stop("Éxito end-to-end incompatible con evaluación/no_solution.")\nattempts$evaluated <- evaluated\nattempts$valid <- evaluated & structural & conservative\nif (any(attempts$valid & (is.na(attempts$search_objective_minutes) | attempts$search_objective_minutes < 0))) stop("Plan válido sin objetivo conservador.")\nattempts$success <- ifelse(attempts$evaluated, attempts$scenario_strict_feasibility, NA_real_)\nno_solution <- !is.na(attempts$evaluation_status) & attempts$evaluation_status == "no_solution" &\n  !is.na(attempts$structural_valid) & !truth(attempts$structural_valid) &\n  !is.na(attempts$solver_status) & attempts$solver_status != "license_pending"\nattempts$success[no_solution] <- 0\nif (any(attempts$success < 0 | attempts$success > 1, na.rm = TRUE)) stop("FR fuera de [0,1].")\n# Filas secundarias/controles incompletas siguen visibles; coprimarios requieren datos íntegros.\nwrite(attempts, "attempt_panel_final.csv")\nprimary <- attempts[attempts$policy == meta[["primary_policy"]], ]\ngeometry_scores <- list(); paired_seeds <- list()\nfor (i in seq_len(nrow(contrasts))) {\n  method <- contrasts$method[[i]]; reference <- contrasts$reference_method[[i]]\n  a <- primary[primary$method == method, ]; b <- primary[primary$method == reference, ]\n  pairing <- c("instance_sha256", "solver_seed")\n  b <- b[match(key(a, pairing), key(b, pairing)), ]\n  if (nrow(a) != nrow(registry) * length(seeds) || anyNA(b$method) || anyNA(a$success) || anyNA(b$success)) stop("Coprimario incompleto: detener inferencia.")\n  score <- as.numeric(a$valid) - as.numeric(b$valid)\n  both <- a$valid & b$valid\n  tolerance <- pmax(tie_abs, tie_rel * pmax(abs(a$search_objective_minutes), abs(b$search_objective_minutes)))\n  difference <- b$search_objective_minutes - a$search_objective_minutes\n  score[both] <- ifelse(abs(difference[both]) <= tolerance[both], 0, sign(difference[both]))\n  # Una comparación emparejada es un estimando operacional con semillas fijas,\n  # no supone que los RNG de métodos distintos generen trayectorias equivalentes.\n  paired <- data.frame(a[c("instance_sha256", "geometry_group_sha256", "n_customers", "solver_seed")],\n    method = method, reference_method = reference, win_score = score,\n    method_valid = a$valid, reference_valid = b$valid,\n    method_success = a$success, reference_success = b$success, fr_difference = a$success - b$success)\n  paired_seeds[[i]] <- paired\n  geometry_scores[[i]] <- do.call(rbind, lapply(split(seq_len(nrow(paired)), paired$geometry_group_sha256), function(ii) {\n    x <- paired[ii, ]\n    if (nrow(x) != length(seeds) || !setequal(x$solver_seed, seeds)) stop("Semillas prefijadas incompletas.")\n    data.frame(x[1, c("geometry_group_sha256", "n_customers", "method", "reference_method")],\n      win_score = mean(x$win_score), fr_difference = mean(x$fr_difference),\n      method_valid_fraction = mean(x$method_valid), reference_valid_fraction = mean(x$reference_valid),\n      expected_seeds = length(seeds), independent_geometry_count = 1L)\n  }))\n}\npaired <- do.call(rbind, paired_seeds); geometry <- do.call(rbind, geometry_scores)\nwrite(paired, "seed_pair_scores.csv"); write(geometry, "geometry_primary_scores.csv")\n# Hoeffding para X_g en [-1,1] y pesos fijados por estrato: sum(w_g)=1.\n# p <= 2 exp(-estimate^2/(2 sum(w_g^2))); intervalos conservadores sin simetría.\nresults <- do.call(rbind, lapply(seq_len(nrow(contrasts)), function(i) {\n  x <- geometry[geometry$method == contrasts$method[[i]] & geometry$reference_method == contrasts$reference_method[[i]], ]\n  counts <- table(factor(x$n_customers, levels = sizes))\n  if (any(counts != quota)) stop("Contraste no cubre todo el test.")\n  weights <- 1 / (length(sizes) * as.numeric(counts[match(x$n_customers, names(counts))]))\n  estimate <- sum(weights * x$win_score); square_sum <- sum(weights^2)\n  radius <- sqrt(2 * square_sum * log(2 / alpha))\n  simultaneous <- sqrt(2 * square_sum * log(2 * nrow(contrasts) / alpha))\n  fr_delta <- sum(weights * x$fr_difference)\n  data.frame(method = contrasts$method[[i]], reference_method = contrasts$reference_method[[i]],\n    geometry_count = nrow(x), estimate = estimate, ci95_lower = max(-1, estimate - radius),\n    ci95_upper = min(1, estimate + radius), simultaneous_ci_lower = max(-1, estimate - simultaneous),\n    simultaneous_ci_upper = min(1, estimate + simultaneous),\n    p_two_sided = min(1, 2 * exp(-estimate^2 / (2 * square_sum))),\n    fr_difference = fr_delta, fr_ci95_lower = max(-1, fr_delta - radius), fr_ci95_upper = min(1, fr_delta + radius),\n    fr_guardrail = if (fr_delta < 0) "observed_degradation_report_explicitly" else "no_observed_degradation_not_noninferiority_proof",\n    scope = "geometry_population_conditional_on_frozen_models_and_finite_seed_set",\n    inference = "stratified_geometry_weighted_hoeffding_mean_bound", sum_squared_geometry_weights = square_sum,\n    minimum_abs_score_for_first_holm_rejection = simultaneous,\n    primary_evidence_is_not_global_robustness = TRUE)\n}))\nresults$p_holm <- p.adjust(results$p_two_sided, method = "holm", n = 2L)\nresults$holm_reject <- results$p_holm <= alpha\nresults$positive_primary_evidence <- results$holm_reject & results$estimate > 0\nresults$negative_primary_evidence <- results$holm_reject & results$estimate < 0\nwrite(results, "coprimary_inference.csv")\n# Secundarios descriptivos; no se ensaya una batería post hoc de p-valores.\nsecondary <- do.call(rbind, lapply(split(seq_len(nrow(attempts)), key(attempts, c("instance_sha256", "policy", "method"))), function(ii) {\n  x <- attempts[ii, ]; valid <- x$valid; known <- sum(!is.na(x$success)); expected <- nrow(x)\n  data.frame(x[1, c("instance_sha256", "geometry_group_sha256", "n_customers", "policy", "method")],\n    expected = expected, evaluated = sum(x$evaluated), conservative_valid = sum(valid), known_success = known,\n    success = if (known == expected) mean(x$success) else NA_real_,\n    success_lower = sum(x$success, na.rm = TRUE) / expected,\n    success_upper = (sum(x$success, na.rm = TRUE) + expected - known) / expected,\n    travel_conditional = mean_available(x$mean_travel_minutes[valid]),\n    cvar_conditional = mean_available(x$cvar_travel_minutes[valid]),\n    duration_conditional = mean_available((x$mean_travel_minutes + x$mean_waiting_minutes)[valid]),\n    recourse_reorders = mean_available(x$mean_recourse_reorders[x$evaluated]),\n    search_seconds = mean_available(x$search_total_seconds), independent_geometry_count = 1L)\n}))\nwrite(secondary, "secondary_descriptive_by_geometry.csv")\nsecondary_lines <- c("# Resultados secundarios descriptivos por tamaño", "",\n  "Métodos en orden fijo. N cuenta geometrías. Costos condicionados a planes conservadores válidos; comparar con cobertura.",\n  "Las cotas FR reflejan desconocimiento cuando falta medición; no son IC poblacionales.",\n  "Para diferencias de costo pareadas con todas las semillas válidas en ambos métodos usar secondary_common_valid_costs.csv.", "",\n  "| n | Política | Método | N | Válidos/intentos | Geometrías con viaje/N | Viaje | Duración | FR [cotas] |",\n  "|---:|---|---|---:|---:|---:|---:|---:|---|")\nfmt <- function(value) if (is.na(value)) "—" else sprintf("%.2f", value)\nfor (size in sizes) for (policy in policies) for (method in methods) {\n  x <- secondary[secondary$n_customers == size & secondary$policy == policy & secondary$method == method, ]\n  secondary_lines <- c(secondary_lines, sprintf("| %s | %s | %s | %d | %d/%d | %d/%d | %s | %s | [%.3f, %.3f] |",\n    size, policy, method, nrow(x), sum(x$conservative_valid), sum(x$expected),\n    sum(!is.na(x$travel_conditional)), nrow(x), fmt(mean_available(x$travel_conditional)),\n    fmt(mean_available(x$duration_conditional)), mean(x$success_lower), mean(x$success_upper)))\n}\nsecondary_lines <- c(secondary_lines, "", "No hay pruebas secundarias post hoc ni ranking global. Los CSV conservan CVaR, reordenamientos y tiempos.",\n  "La memoria se informa sólo cuando hay medición: CUDA del proceso durante el intento y RSS máximo histórico del proceso; no son incrementos aislados por método. Entrenamiento/validación aparecen por checkpoint en offline_model_costs.csv.")\nwriteLines(secondary_lines, file.path(output, "secondary_results.md"), useBytes = TRUE)\n\n# Costos offline se muestran una vez por checkpoint, incluido el rechazado; HBG\n# sin feromona reutiliza esos pesos y no duplica el cargo de entrenamiento.\noffline <- read("offline_model_costs.csv")\nneed(offline, c("method", "solver_seed", "model_checkpoint_sha256", "training_seconds_including_validation", "validation_seconds_already_included"))\nif (anyDuplicated(key(offline, c("method", "solver_seed")))) stop("Costo offline duplicado por semilla/modelo.")\nwrite(offline, "offline_model_costs.csv")\nfor (field in c("evaluation_seconds", "trace_validation_seconds", "peak_cuda_allocated_bytes", "process_max_rss_kib")) {\n  if (!field %in% names(attempts)) attempts[[field]] <- NA_real_ else attempts[[field]] <- num(attempts[[field]])\n}\nsearch_keys <- c("instance_sha256", "method", "solver_seed")\nfor (ii in split(seq_len(nrow(attempts)), key(attempts, search_keys))) {\n  values <- attempts$search_total_seconds[ii]\n  if (length(unique(values)) != 1L) stop("Tiempo de búsqueda cambia entre políticas de un mismo intento.")\n}\nsearch_once <- attempts[!duplicated(key(attempts, search_keys)), ]\ncompute <- do.call(rbind, lapply(methods, function(method) {\n  x <- search_once[search_once$method == method, ]; evaluations <- attempts[attempts$method == method, ]\n  total_if_complete <- function(values) if (anyNA(values)) NA_real_ else sum(values)\n  data.frame(method = method, search_attempts = nrow(x),\n    search_seconds = total_if_complete(x$search_total_seconds),\n    evaluated_policy_rows = sum(evaluations$evaluated),\n    evaluation_time_rows_available = sum(!is.na(evaluations$evaluation_seconds)),\n    evaluation_seconds = total_if_complete(evaluations$evaluation_seconds),\n    trace_validation_time_rows_available = sum(!is.na(evaluations$trace_validation_seconds)),\n    trace_validation_seconds = total_if_complete(evaluations$trace_validation_seconds),\n    training_is_in_separate_shared_checkpoint_table = TRUE,\n    missing_time_is_unknown_not_zero = TRUE,\n    cuda_peak_rows_available = sum(!is.na(x$peak_cuda_allocated_bytes)),\n    cuda_process_peak_during_attempt_bytes = if (all(is.na(x$peak_cuda_allocated_bytes))) NA_real_ else max(x$peak_cuda_allocated_bytes, na.rm = TRUE),\n    cpu_lifetime_high_water_rows_available = sum(!is.na(x$process_max_rss_kib)),\n    cpu_process_lifetime_high_water_kib = if (all(is.na(x$process_max_rss_kib))) NA_real_ else max(x$process_max_rss_kib, na.rm = TRUE),\n    memory_scope = "process_memory_not_isolated_method_increment;_training_memory_not_measured")\n}))\nwrite(compute, "compute_costs_without_policy_search_duplication.csv")\n\n# Costos secundarios sólo sobre la misma geometría con todas las semillas válidas\n# en ambos métodos; cada exclusión conserva su fila, denominador y motivo.\ncost_pairs <- unique(rbind(contrasts, data.frame(method = setdiff(methods, "gurobi"), reference_method = "gurobi")))\npaired_costs <- list(); counter <- 0L\nfor (i in seq_len(nrow(cost_pairs))) for (policy in policies) for (geometry_id in registry$geometry_group_sha256) {\n  a <- attempts[attempts$geometry_group_sha256 == geometry_id & attempts$policy == policy & attempts$method == cost_pairs$method[[i]], ]\n  b <- attempts[attempts$geometry_group_sha256 == geometry_id & attempts$policy == policy & attempts$method == cost_pairs$reference_method[[i]], ]\n  b <- b[match(a$solver_seed, b$solver_seed), ]\n  for (metric in c("search_objective_minutes", "mean_travel_minutes", "cvar_travel_minutes", "duration_minutes")) {\n    av <- if (metric == "duration_minutes") a$mean_travel_minutes + a$mean_waiting_minutes else a[[metric]]\n    bv <- if (metric == "duration_minutes") b$mean_travel_minutes + b$mean_waiting_minutes else b[[metric]]\n    eligible <- nrow(a) == length(seeds) && nrow(b) == length(seeds) && all(a$valid) && all(b$valid) && !anyNA(av) && !anyNA(bv)\n    counter <- counter + 1L\n    paired_costs[[counter]] <- data.frame(geometry_group_sha256 = geometry_id,\n      n_customers = registry$n_customers[match(geometry_id, registry$geometry_group_sha256)], policy = policy,\n      method = cost_pairs$method[[i]], reference_method = cost_pairs$reference_method[[i]], metric = metric,\n      expected_seeds = length(seeds), method_valid = sum(a$valid), reference_valid = sum(b$valid),\n      status = if (eligible) "common_complete_valid_geometry" else "excluded_incomplete_or_invalid",\n      method_mean = if (eligible) mean(av) else NA_real_, reference_mean = if (eligible) mean(bv) else NA_real_,\n      delta = if (eligible) mean(av) - mean(bv) else NA_real_, independent_geometry_count = 1L,\n      scope = "secondary_descriptive_conditional_common_complete_valid_geometry")\n  }\n}\nwrite(do.call(rbind, paired_costs), "secondary_common_valid_costs.csv")\n# Cota conservadora frente a costo conservador, sin restarla al viaje estocástico.\nif ("best_bound" %in% names(attempts)) {\n  attempts$best_bound <- num(attempts$best_bound)\n  gaps <- lapply(seq_len(nrow(attempts)), function(i) {\n    x <- attempts[i, ]; exact <- attempts[attempts$instance_sha256 == x$instance_sha256 & attempts$method == "gurobi", ]\n    bound <- if (all(is.na(exact$best_bound))) NA_real_ else max(exact$best_bound, na.rm = TRUE)\n    objective <- if (x$valid) x$search_objective_minutes else NA_real_\n    compatible <- !is.na(bound) && !is.na(objective) && objective > 0 && bound <= objective + max(tie_abs, tie_rel * abs(objective))\n    data.frame(x[c("instance_sha256", "geometry_group_sha256", "n_customers", "policy", "method", "solver_seed")],\n      conservative_objective_minutes = objective, gurobi_conservative_bound_minutes = bound,\n      gap_upperbound_pct = if (compatible) 100 * max(0, objective - bound) / objective else NA_real_,\n      status = if (compatible) "available_conservative_bound_gap" else "unavailable_zero_or_inconsistent",\n      bound_envelope_is_not_single_attempt_runtime = TRUE, not_stochastic_optimality = TRUE,\n      gap_denominator = "candidate_conservative_objective_not_unknown_optimum")\n  })\n  write(do.call(rbind, gaps), "secondary_conservative_gap_audit.csv")\n}\n\nlines <- c("# Comparación confirmatoria de geometrías independientes", "",\n  paste("Release:", meta[["final_release_id"]]), "",\n  "Win-score: factibilidad conservadora primero; después costo conservador; ambos fallidos empatan.",\n  "Se promedian las semillas prefijadas dentro de geometría, con peso igual de los cuatro estratos.",\n  "Los IC y p-valores Hoeffding son conservadores y suponen independencia de geometrías. Holm controla los dos coprimarios.",\n  "HBG sin feromona aísla sólo la memoria de feromona EN INFERENCIA con los mismos pesos entrenados; no identifica efectos de TB/DB/replay ni ausencia de feromona durante entrenamiento.",\n  "La guardia FR registra degradación observada; no se ha fijado un margen de no inferioridad y no prueba ausencia de daño.", "",\n  "| Método - control | N geometrías | Efecto | IC95% | IC simultáneo | p Holm | Diferencia FR | Evidencia positiva |",\n  "|---|---:|---:|---|---|---:|---:|---|")\nfor (i in seq_len(nrow(results))) {\n  x <- results[i, ]\n  lines <- c(lines, sprintf("| %s - %s | %d | %.4f | [%.4f, %.4f] | [%.4f, %.4f] | %.5f | %.4f | %s |",\n    x$method, x$reference_method, x$geometry_count, x$estimate, x$ci95_lower, x$ci95_upper,\n    x$simultaneous_ci_lower, x$simultaneous_ci_upper, x$p_holm, x$fr_difference, x$positive_primary_evidence))\n}\nlines <- c(lines, "", "No significancia no demuestra equivalencia. Estos datos no identifican optimalidad estocástica, c+Q ni superioridad general.",\n  "El alcance es la población generada con condicionamiento de ventanas y reglas selladas, condicionado a modelos congelados y tres semillas de solver.",\n  "Fuentes: Hoeffding (1963), https://doi.org/10.1080/01621459.1963.10500830; R stats::p.adjust, https://search.r-project.org/R/refmans/stats/html/p.adjust.html")\nwriteLines(lines, file.path(output, "confirmatory_results.md"), useBytes = TRUE)\ndraw <- function() {\n  par(mar = c(5, 12, 4, 1), las = 1)\n  y <- seq_len(nrow(results))\n  plot(results$estimate, y, xlim = c(-1, 1), ylim = c(.5, nrow(results) + .5), yaxt = "n", pch = 19,\n       xlab = "Win-score medio con IC simultaneo Hoeffding", ylab = "", main = "Dos contrastes coprimarios; N por geometria")\n  axis(2, y, paste0(results$method, " - ", results$reference_method, " (N=", results$geometry_count, ")"), cex.axis = .7)\n  segments(results$simultaneous_ci_lower, y, results$simultaneous_ci_upper, y, lwd = 2)\n  abline(v = 0, lty = 2, col = "#777777")\n}\npdf(file.path(output, "coprimary_effects.pdf"), width = 11, height = 5); draw(); dev.off()\npng(file.path(output, "coprimary_effects.png"), width = 3300, height = 1500, res = 300); draw(); dev.off()\nverify()\nwriteLines(capture.output(sessionInfo()), file.path(output, "sessionInfo.txt"))\nwrite(data.frame(check = c("source_hashes_before_after", "all_scheduled_attempts_present", "complete_primary_seed_pairs",\n                          "independent_geometry_quotas", "two_coprimaries_holm", "no_secondary_pvalue_search"), passed = TRUE), "processing_checks.csv")\ncat("R: inferencia coprimaria completa; el notebook debe sellar el reporte.\\n")\n'
if ANALYSIS_SOURCE not in ('development06', 'final10'):
    raise ValueError('Fuente de análisis desconocida.')
if ANALYSIS_SOURCE == 'final10':
    if not FINAL_REPORT_RELATIVE_PATH or len(FINAL_REPORT_CHECKSUMS_SHA256) != 64 or not SELECTION_REASON.strip():
        raise ValueError('Fijar ruta, hash y motivo del release final; no seleccionar por resultados.')
    drive.mount('/content/drive')
    def final_sha(path):
        result = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                result.update(block)
        return result.hexdigest()
    def final_safe(root, relative):
        root = Path(root).resolve(); relative = Path(relative)
        if relative.is_absolute() or '..' in relative.parts or not relative.parts:
            raise ValueError('Ruta final inválida.')
        result = root / relative
        if not result.resolve().is_relative_to(root):
            raise ValueError('Ruta fuera del directorio previsto.')
        return result
    FINAL_REPORT = final_safe(DRIVE_ROOT, FINAL_REPORT_RELATIVE_PATH)
    FINAL_WORK = Path('/content/final08_' + uuid.uuid4().hex[:12])
    PACKAGE = FINAL_WORK / 'code/svrplab'
    PACKAGE.mkdir(parents=True)
    for name, source in FINAL_MODULES.items():
        (PACKAGE / name).write_text(source, encoding='utf-8')
    sys.path.insert(0, str(PACKAGE.parent))
    for name in list(sys.modules):
        if name == 'svrplab' or name.startswith('svrplab.'):
            del sys.modules[name]
    import importlib
    importlib.invalidate_caches()
    from svrplab.final_experiment import verify_analysis_export
    final_contract = verify_analysis_export(FINAL_REPORT, FINAL_REPORT_CHECKSUMS_SHA256)
    final_design = final_contract['design']
    if (final_contract['schema_version'] != 'final-analysis-export-v1' or
        final_contract['role'] != 'confirmatory_holdout' or final_contract['analysis_mode'] != 'confirmatory'):
        raise ValueError('El release no corresponde a la rama confirmatoria.')
    final_r_sha256 = hashlib.sha256(FINAL_R_SOURCE.encode()).hexdigest()
    if (final_contract['analysis_r_sha256'] != final_r_sha256 or
        final_contract['analysis_config_sha256'] != ANALYSIS_CONFIG_SHA256):
        raise ValueError('R cambió después de congelar el análisis. Usar exactamente la versión sellada, sin retocar hipótesis.')
    if (final_design['primary_inference']['method'] != 'stratified_geometry_weighted_hoeffding_mean_bound' or
        final_design['primary_multiplicity'] != 'holm_two_sided_familywise_0.05' or
        final_design['primary_endpoint'] != 'geometry_mean_feasibility_first_conservative_cost_win_score' or
        final_design['win_score']['seed_aggregation'] != 'mean_all_prefixed_seed_pairs_within_geometry' or
        final_design['win_score']['size_aggregation'] != 'equal_weight_four_size_strata' or
        final_design['primary_inference']['noninferiority_margin_fr'] is not None):
        raise ValueError('Estimando/multiplicidad/guardia distintos del análisis implementado.')
    FINAL_INPUT = FINAL_WORK / 'input'
    FINAL_OUTPUT = FINAL_WORK / 'output'
    FINAL_INPUT.mkdir()
    final_manifest = json.loads((FINAL_REPORT / 'CHECKSUMS.json').read_text())
    core = ['attempts.csv', 'registry.csv', 'research_design.json', 'final_release.json',
            'analysis_export_contract.json', 'runtime_summary.csv', 'runtime_environments.json']
    if not set(core) <= set(final_manifest):
        raise ValueError('Exporte final incompleto: faltan datos, contrato o evidencia sellada de runtime.')
    core += [name for name in final_manifest if name.startswith('source_manifests/')]
    core_hashes = {}
    for name in core:
        target = final_safe(FINAL_INPUT, name)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(final_safe(FINAL_REPORT, name), target)
        if final_sha(target) != final_manifest[name]:
            raise ValueError('Copia de entrada final inconsistente: ' + name)
        core_hashes[name] = final_manifest[name]
    final_release = json.loads((FINAL_INPUT / 'final_release.json').read_text())
    runtime_environments = json.loads((FINAL_INPUT / 'runtime_environments.json').read_text())
    runtime_count = len(runtime_environments)
    timing_stratification = final_release.get('timing_requires_hardware_stratification')
    if (not isinstance(runtime_environments, dict) or not runtime_environments
        or final_release.get('distinct_runtime_environments') != runtime_count
        or type(timing_stratification) is not bool or timing_stratification != (runtime_count > 1)):
        raise ValueError('Inventario de runtimes y bandera temporal del release inconsistentes.')
    timing_note = ('El release registra varios entornos de ejecución: las comparaciones temporales requieren '
        'separar resultados por hardware/runtime. ' if timing_stratification else
        'El release registra un único entorno de ejecución. ')
    timing_note += ('Los tiempos son descriptivos del cómputo observado; no estiman una aceleración causal '
        '(speedup) del método. runtime_summary.csv y runtime_environments.json se conservan sellados '
        'en source_evidence.')
    print(timing_note)
    def final_csv(path, rows, fields):
        with Path(path).open('w', newline='', encoding='utf-8') as stream:
            writer = csv.DictWriter(stream, fieldnames=fields)
            writer.writeheader(); writer.writerows(rows)
    final_bridge = {key: final_contract[key] for key in ['schema_version', 'role', 'analysis_mode', 'comparison_id', 'final_release_id']}
    final_bridge.update(primary_inference=final_design['primary_inference']['method'], familywise_alpha=.05,
        primary_policy=final_design['primary_policy'],
        cost_tie_absolute_minutes=final_design['win_score']['cost_tie_absolute_minutes'],
        cost_tie_relative=final_design['win_score']['cost_tie_relative'],
        test_geometries_per_size=final_design['geometry_quotas_per_size']['test'])
    final_csv(FINAL_INPUT / 'analysis_bridge.csv', [{'key': k, 'value': v} for k, v in final_bridge.items()], ['key', 'value'])
    final_csv(FINAL_INPUT / 'primary_contrasts.csv', [{'method': m, 'reference_method': r}
        for m, r in final_design['primary_contrasts']], ['method', 'reference_method'])
    for filename, field, values in [('expected_methods.csv', 'method', final_contract['expected_methods']),
        ('expected_seeds.csv', 'solver_seed', final_contract['solver_seeds']),
        ('expected_policies.csv', 'policy', final_contract['policies']),
        ('expected_sizes.csv', 'n_customers', final_design['sizes'])]:
        final_csv(FINAL_INPUT / filename, [{field: str(v)} for v in values], [field])
    offline_rows = []
    for model in final_contract['source_chain']['freeze']['models']:
        summary = model['summary']
        offline_rows.append({'method': model['method'], 'solver_seed': model['seed'],
            'model_checkpoint_sha256': summary['model_checkpoint_sha256'],
            'selection_status': summary['selection_status'],
            'training_seconds_including_validation': summary.get('training_seconds'),
            'validation_seconds_already_included': summary.get('validation_seconds'),
            'shared_with': 'hbg_no_pheromone' if model['method'] == 'hbg_facs' else ''})
    final_csv(FINAL_INPUT / 'offline_model_costs.csv', offline_rows,
        ['method', 'solver_seed', 'model_checkpoint_sha256', 'selection_status',
         'training_seconds_including_validation', 'validation_seconds_already_included', 'shared_with'])
    final_csv(FINAL_INPUT / 'input_file_hashes.csv', [{'relative_path': k, 'sha256': v}
        for k, v in core_hashes.items()], ['relative_path', 'sha256'])
    Path('/content/.svrplab_final08_verified').write_text(FINAL_REPORT_CHECKSUMS_SHA256)
    FINAL_R_SCRIPT = FINAL_WORK / 'confirmatory_results_08.R'
    FINAL_R_SCRIPT.write_text(FINAL_R_SOURCE, encoding='utf-8')
    if shutil.which('Rscript') is None:
        subprocess.run(['apt-get', '-qq', 'update'], check=True)
        subprocess.run(['apt-get', '-qq', 'install', '-y', 'r-base-core'], check=True)
    result = subprocess.run(['Rscript', '--vanilla', str(FINAL_R_SCRIPT), str(FINAL_INPUT), str(FINAL_OUTPUT)],
        env=dict(os.environ, SVRPLAB_COLAB_ANALYSIS='1'), text=True, capture_output=True)
    (FINAL_WORK / 'R_stdout.log').write_text(result.stdout)
    (FINAL_WORK / 'R_stderr.log').write_text(result.stderr)
    print(result.stdout)
    if result.returncode:
        print(result.stderr)
        raise RuntimeError('Inferencia final detenida; fuentes y logs conservados, sin reporte validado.')
    # Confirmar de nuevo cadena íntegra al terminar; R no modifica datos fuente.
    verify_analysis_export(FINAL_REPORT, FINAL_REPORT_CHECKSUMS_SHA256)
    final_processing_id = 'final_analysis08_' + hashlib.sha256((FINAL_REPORT_CHECKSUMS_SHA256 + final_r_sha256).encode()).hexdigest()[:20]
    final_identity = {'schema_version': 'final-analysis-08-v1', 'processing_id': final_processing_id,
        'input_report_checksums_sha256': FINAL_REPORT_CHECKSUMS_SHA256, 'input_report_relative_path': FINAL_REPORT_RELATIVE_PATH,
        'final_release_id': final_contract['final_release_id'], 'comparison_id': final_contract['comparison_id'],
        'analysis_r_sha256': final_r_sha256, 'selection_reason': SELECTION_REASON,
        'source_chain_verified_before_and_after': True, 'role': 'confirmatory_holdout',
        'distinct_runtime_environments': runtime_count,
        'timing_requires_hardware_stratification': timing_stratification,
        'created_utc': datetime.now(timezone.utc).isoformat()}
    (FINAL_OUTPUT / 'identity.json').write_text(json.dumps(final_identity, indent=2, ensure_ascii=False) + '\n')
    (FINAL_OUTPUT / 'input_manifest.json').write_text(json.dumps(final_manifest, indent=2) + '\n')
    for source in [FINAL_R_SCRIPT, FINAL_WORK / 'R_stdout.log', FINAL_WORK / 'R_stderr.log']:
        shutil.copy2(source, FINAL_OUTPUT / source.name)
    for name in core:
        destination = final_safe(FINAL_OUTPUT / 'source_evidence', name)
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(final_safe(FINAL_INPUT, name), destination)
    (FINAL_OUTPUT / 'timing_interpretation.md').write_text(timing_note + '\n', encoding='utf-8')
    for name in ['confirmatory_results.md', 'secondary_results.md']:
        document = FINAL_OUTPUT / name
        document.write_text(document.read_text(encoding='utf-8').rstrip() + '\n\n' + timing_note + '\n', encoding='utf-8')
    final_output_manifest = {str(path.relative_to(FINAL_OUTPUT)): final_sha(path)
        for path in sorted(FINAL_OUTPUT.rglob('*')) if path.is_file()}
    (FINAL_OUTPUT / 'CHECKSUMS.json').write_text(json.dumps(final_output_manifest, indent=2, sort_keys=True) + '\n')
    suffix = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:6]
    FINAL_DESTINATION = DRIVE_ROOT / 'final_analysis_runs' / final_contract['final_release_id'] / final_processing_id / suffix
    shutil.copytree(FINAL_OUTPUT, FINAL_DESTINATION)
    for name, digest in final_output_manifest.items():
        if final_sha(FINAL_DESTINATION / name) != digest:
            raise ValueError('Copia final a Drive inconsistente.')
    if final_sha(FINAL_DESTINATION / 'CHECKSUMS.json') != final_sha(FINAL_OUTPUT / 'CHECKSUMS.json'):
        raise ValueError('Manifiesto final a Drive inconsistente.')
    print('Reporte confirmatorio sellado:', FINAL_DESTINATION)
    print('SHA-256 CHECKSUMS:', final_sha(FINAL_DESTINATION / 'CHECKSUMS.json'))
    from IPython.display import display, Markdown, Image
    display(Markdown((FINAL_DESTINATION / 'confirmatory_results.md').read_text()))
    display(Markdown((FINAL_DESTINATION / 'secondary_results.md').read_text()))
    display(Image(filename=str(FINAL_DESTINATION / 'coprimary_effects.png'), width=1000))
